<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 55%,#00A86A 100%);border-radius:20px;padding:38px 40px 34px 40px;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#F5C242;font-size:12px;letter-spacing:3px;font-weight:700;">BANQUE AFRICAINE DE DÉVELOPPEMENT &nbsp;·&nbsp; UA STATAFRIC &nbsp;·&nbsp; GTS17</div><div style="color:#fff;font-size:2.15em;font-weight:800;margin-top:10px;line-height:1.12;">Lumières nocturnes&nbsp;: collecte des données<br>NASA Black Marble (VNP46)</div><div style="color:#E6F6EE;font-size:1.05em;font-style:italic;margin-top:14px;max-width:62em;line-height:1.6;">Une chaîne d’acquisition reproductible, pilotée par un seul paramètre&nbsp;— le code ISO3 du pays&nbsp;— qui interroge le catalogue NASA&nbsp;CMR, teste les quatre produits Black Marble sur des périodes fixées, puis télécharge et vérifie les granules retenus.</div><div style="margin-top:22px;height:4px;width:130px;background:#F5C242;border-radius:2px;"></div><div style="color:#CFEEDE;font-size:.92em;margin-top:16px;">Atelier technique <b>« Emerging Issues, Emerging Practice »</b> &nbsp;·&nbsp; Jour 4 — Travaux pratiques, partie 1 «&nbsp;Collecter&nbsp;»<br>Compatible <b>Google Colab</b>, <b>Kaggle</b> et <b>Jupyter local</b> (Windows, macOS, Linux)</div></div>

## Ce que fait ce notebook

Vous renseignez **un seul paramètre** — le code ISO3 du pays (`CIV`, `SDN`, `RWA`, `TUN`…) — et le notebook s'occupe du reste : il déduit l'emprise géographique du pays, calcule les tuiles VIIRS qui la recouvrent, interroge le catalogue de la NASA pour les quatre produits Black Marble sur des périodes fixées, établit un inventaire de disponibilité, puis télécharge un échantillon vérifié.

| # | Étape | Réseau | Identifiants NASA |
|---|---|---|---|
| 1 | Détection de l'environnement d'exécution et des dossiers de travail | non | non |
| 2 | Vérification et installation des dépendances | pip | non |
| 3 | Récupération du jeton Earthdata (4 sources possibles) | non | — |
| 4 | Paramètres de la collecte : **pays, produits, périodes** | non | non |
| 5 | Emprise nationale et tuiles VIIRS correspondantes | non | non |
| 6 | Fonctions d'interrogation du catalogue NASA CMR | non | non |
| 7 | **Test de disponibilité des quatre produits** | oui | non |
| 8 | Téléchargement résumable et vérifié | oui | **oui** |
| 9 | Contrôle qualité : ouverture d'un fichier HDF5 et aperçu | non | non |

Les étapes 1 à 7 fonctionnent **sans aucun identifiant** : la recherche dans le catalogue NASA est publique. Le jeton n'est nécessaire qu'au moment du téléchargement effectif des fichiers, à l'étape 8. Vous pouvez donc faire tourner l'essentiel du notebook, et toute la partie pédagogique, avant même d'avoir créé un compte.

<div style="background:#F4F7F5;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">🔑 Mode d'emploi en trois gestes</b><div style="color:#231F20;margin-top:5px;line-height:1.55;"><b>1.</b> Exécutez les cellules dans l'ordre (menu <i>Exécution → Tout exécuter</i>). <b>2.</b> Modifiez uniquement la cellule de l'étape 4 — c'est le seul endroit à éditer. <b>3.</b> Relancez. Le notebook est <i>idempotent</i> : les fichiers déjà téléchargés et valides sont ignorés, les téléchargements interrompus reprennent où ils s'étaient arrêtés.</div></div>

---

## Rappel : que sont les données Black Marble ?

Le capteur **VIIRS/DNB** (*Day–Night Band*), embarqué sur les satellites Suomi-NPP et NOAA-20, mesure chaque nuit la lumière émise vers l'espace depuis la surface terrestre. La NASA transforme ces mesures brutes en une suite de produits corrigés appelée **Black Marble** (collection VNP46), où l'effet de la Lune, de l'atmosphère, des nuages et de la neige a été retiré. L'unité est le **nW·cm⁻²·sr⁻¹** (nanowatt par centimètre carré et par stéradian).

Quatre produits se distinguent uniquement par leur **fréquence temporelle** — ils partagent la même grille, les mêmes tuiles et la même structure de fichier :

| Produit | Fréquence | Un fichier couvre | Variable principale | Usage typique |
|---|---|---|---|---|
| `VNP46A1` | quotidien | une nuit, radiance brute au capteur | `DNB_At_Sensor_Radiance_500m` | diagnostic, contrôle de la donnée source |
| `VNP46A2` | quotidien | une nuit, corrigée BRDF lunaire et comblée | `Gap_Filled_DNB_BRDF-Corrected_NTL` | suivi de crise, coupures d'électricité |
| `VNP46A3` | mensuel | un mois calendaire (composite) | `NearNadir_Composite_Snow_Free` | saisonnalité, séries infra-annuelles |
| `VNP46A4` | annuel | une année calendaire (composite) | `NearNadir_Composite_Snow_Free` | séries longues, comparaisons régionales |

**La règle des tuiles.** Le globe est découpé en cellules fixes de 10° × 10° en latitude/longitude, indexées `h` (colonne, 0 à 35, d'ouest en est) et `v` (ligne, 0 à 17, du nord au sud). Chaque fichier correspond à **une tuile et une date**. Un pays se traduit donc par un jeu fixe de tuiles : la Côte d'Ivoire tient dans `h17v07` et `h17v08`, le Soudan en réclame six. L'étape 5 calcule ce jeu automatiquement à partir de l'emprise du pays.

**La règle du nommage.** Un granule s'appelle par exemple `VNP46A4.A2023001.h17v08.001.2024031102344.h5`, soit `produit . A + année + jour julien . tuile . version . horodatage de traitement . h5`. L'horodatage de traitement étant **imprévisible**, on ne construit jamais une URL à la main : on interroge le catalogue, qui renvoie les vrais noms.

<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ Pourquoi passer par le catalogue CMR plutôt que par le répertoire LAADS</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">Le <b>CMR</b> (<i>Common Metadata Repository</i>) est le moteur de recherche unifié de la NASA. Une requête suffit pour demander <i>« tous les granules du produit X qui intersectent cette emprise entre ces deux dates »</i> — c'est le catalogue qui fait le travail d'intersection géographique, et il renvoie en prime la taille de chaque fichier, ce qui permet d'estimer le volume <b>avant</b> de télécharger quoi que ce soit. La recherche est publique et ne demande aucun jeton.</div></div>

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 01 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Environnement d'exécution</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Colab, Kaggle ou local — le notebook s'adapte tout seul</div></div>

Le seul point réellement délicat de la portabilité, ce sont les **chemins de fichiers**. Un chemin `C:/NTL_LOCAL` codé en dur ne survit ni à Colab, ni à Kaggle, ni à un collègue sous macOS. La cellule ci-dessous détecte la plateforme, choisit une racine de travail accessible en écriture, et la teste réellement avant de la retenir.

Vous pouvez imposer votre propre dossier en définissant la variable d'environnement `NTL_HOME` avant de lancer Jupyter.

In [ ]:
# =============================================================================
#  ÉTAPE 1 — Environnement d'exécution, dossiers de travail, encodage console
# =============================================================================
import os, sys, platform, shutil, tempfile, datetime as dt
from pathlib import Path

# --- 1.1  Console : forcer l'UTF-8 quand c'est possible ----------------------
# Sous Windows, une console héritée en cp1252 fait planter un simple print("✓").
# On tente de reconfigurer la sortie ; si c'est impossible, on bascule sur des
# symboles ASCII. Le notebook reste lisible partout.
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass


def _symboles():
    jeu = {"ok": "\u2713", "ko": "\u2717", "att": "\u26a0", "fl": "\u2192", "pt": "\u2022"}
    try:
        "".join(jeu.values()).encode(getattr(sys.stdout, "encoding", None) or "utf-8")
        return jeu
    except Exception:
        return {"ok": "[OK]", "ko": "[X]", "att": "[!]", "fl": "->", "pt": "-"}


SYM = _symboles()
LARGEUR = 78


def titre(texte):
    """Affiche un titre de section encadré, largeur fixe, sans dépendance."""
    print("\n" + "=" * LARGEUR)
    print("  " + texte.upper())
    print("=" * LARGEUR)


def ligne(cle, valeur, symbole=" "):
    print(f"  {symbole} {cle:<30} {valeur}")


# --- 1.2  Quelle plateforme ? ------------------------------------------------
def detecter_plateforme():
    """Renvoie 'Colab', 'Kaggle' ou 'Local'. L'ordre des tests compte."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "Kaggle"
    if "google.colab" in sys.modules or Path("/content").exists():
        return "Colab"
    return "Local"


PLATEFORME = detecter_plateforme()


# --- 1.3  Une racine de travail réellement accessible en écriture ------------
def choisir_racine(plateforme):
    """Essaie plusieurs emplacements et retourne le premier où l'on peut écrire."""
    candidats = []
    if os.environ.get("NTL_HOME"):
        candidats.append(Path(os.environ["NTL_HOME"]))
    if plateforme == "Colab":
        candidats.append(Path("/content/NTL_LOCAL"))
    elif plateforme == "Kaggle":
        candidats.append(Path("/kaggle/working/NTL_LOCAL"))
    candidats += [Path.home() / "NTL_LOCAL",
                  Path.cwd() / "NTL_LOCAL",
                  Path(tempfile.gettempdir()) / "NTL_LOCAL"]

    for c in candidats:
        try:
            c.mkdir(parents=True, exist_ok=True)
            sonde = c / ".test_ecriture"
            sonde.write_text("ok", encoding="utf-8")
            sonde.unlink()
            return c
        except Exception:
            continue
    raise RuntimeError("Aucun dossier accessible en écriture. Définissez NTL_HOME.")


RACINE = choisir_racine(PLATEFORME)
DOSSIERS = {
    "cache_h5":  RACINE / "h5_cache",     # les granules .h5 téléchargés
    "catalogue": RACINE / "catalogue",    # les inventaires CSV produits ici
    "figures":   RACINE / "figures",      # les graphiques exportés
    "journaux":  RACINE / "journaux",     # les traces d'exécution
}
for d in DOSSIERS.values():
    d.mkdir(parents=True, exist_ok=True)

# --- 1.4  Rapport ------------------------------------------------------------
_libre = shutil.disk_usage(RACINE).free / 1e9
SESSION_DEBUT = dt.datetime.now()

titre("Environnement d'exécution")
ligne("Plateforme détectée", PLATEFORME, SYM["ok"])
ligne("Système", f"{platform.system()} {platform.release()} ({platform.machine()})", SYM["pt"])
ligne("Python", platform.python_version(), SYM["pt"])
ligne("Racine de travail", str(RACINE), SYM["ok"])
for nom, chemin in DOSSIERS.items():
    ligne(f"  dossier {nom}", str(chemin.relative_to(RACINE)), SYM["pt"])
ligne("Espace disque libre", f"{_libre:,.1f} Go".replace(",", " "),
      SYM["ok"] if _libre > 5 else SYM["att"])
ligne("Horodatage", SESSION_DEBUT.strftime("%Y-%m-%d %H:%M:%S"), SYM["pt"])

if _libre < 5:
    print(f"\n  {SYM['att']} Moins de 5 Go disponibles. Les produits quotidiens sont volumineux :")
    print("     restez sur VNP46A4 (annuel) ou réduisez MAX_FICHIERS à l'étape 4.")

<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ Colab : conserver les fichiers entre deux sessions</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">La machine Colab est effacée à la fermeture. Pour garder les granules, montez votre Google Drive <i>avant</i> d'exécuter la cellule ci-dessus et pointez la racine dessus :<br><code>from google.colab import drive; drive.mount('/content/drive')</code><br><code>import os; os.environ['NTL_HOME'] = '/content/drive/MyDrive/NTL_LOCAL'</code></div></div>

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 02 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Dépendances</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Deux bibliothèques indispensables, trois optionnelles</div></div>

Le socle est volontairement minimal : `requests` pour le réseau et `pandas` pour les tableaux. Tout le reste est optionnel et ne bloque jamais l'exécution — `h5py` sert à ouvrir les fichiers HDF5 à l'étape 9, `numpy` et `matplotlib` à les visualiser. Aucune bibliothèque géospatiale lourde (`gdal`, `rasterio`, `geopandas`) n'est requise : c'est ce qui rend ce notebook installable en trente secondes sur n'importe quelle machine.

La cellule n'installe **que ce qui manque**, ce qui évite de casser un environnement local déjà configuré.

In [ ]:
# =============================================================================
#  ÉTAPE 2 — Dépendances : vérifier, installer ce qui manque, rapporter
# =============================================================================
import importlib, importlib.util, subprocess

REQUIS     = {"requests": "requests", "pandas": "pandas"}
OPTIONNELS = {"h5py": "h5py", "numpy": "numpy", "matplotlib": "matplotlib"}


def installer(paquets):
    """Installation silencieuse via le Python courant (fonctionne aussi hors Jupyter).

    Trois tentatives successives : installation normale, puis --user, puis
    --break-system-packages. La dernière est nécessaire sur les Linux récents
    (Debian, Ubuntu 23.04+) qui refusent pip sur le Python du système (PEP 668).
    """
    base = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    for variante in ([], ["--user"], ["--break-system-packages"]):
        try:
            subprocess.check_call(base + variante + list(paquets),
                                  stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            return True
        except Exception:
            continue
    return False


def assurer(catalogue, bloquant):
    """Installe les modules absents puis renvoie {module: version | None}."""
    manquants = [pkg for mod, pkg in catalogue.items() if importlib.util.find_spec(mod) is None]
    if manquants:
        print(f"  {SYM['fl']} Installation de : {', '.join(manquants)} ...")
        if not installer(manquants):
            msg = "installation impossible (pas d'accès réseau vers PyPI ?)"
            if bloquant:
                raise RuntimeError(f"Dépendances requises absentes : {manquants} — {msg}")
            print(f"  {SYM['att']} {msg} — on continue sans.")
        importlib.invalidate_caches()

    etat = {}
    for mod in catalogue:
        try:
            m = importlib.import_module(mod)
            etat[mod] = getattr(m, "__version__", "?")
        except Exception:
            etat[mod] = None
    return etat


titre("Dépendances")
etat_requis = assurer(REQUIS, bloquant=True)
etat_option = assurer(OPTIONNELS, bloquant=False)

for mod, ver in etat_requis.items():
    ligne(f"{mod} (requis)", ver, SYM["ok"])
for mod, ver in etat_option.items():
    ligne(f"{mod} (optionnel)", ver or "absent",
          SYM["ok"] if ver else SYM["att"])

import requests
import pandas as pd

# Drapeaux utilisés plus loin pour désactiver proprement ce qui n'est pas installé
H5PY_DISPO = etat_option.get("h5py") is not None
MPL_DISPO  = etat_option.get("matplotlib") is not None and etat_option.get("numpy") is not None

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 03 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Identifiants NASA Earthdata</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Un jeton, quatre façons de le fournir — jamais écrit dans le notebook</div></div>

Les fichiers Black Marble sont **gratuits**, mais leur téléchargement exige un compte NASA Earthdata. L'authentification se fait par **jeton porteur** (*bearer token*), pas par mot de passe : une longue chaîne de caractères que l'on transmet dans l'en-tête HTTP `Authorization: Bearer <jeton>`.

**Obtenir un jeton, une seule fois :**

1. Créez un compte gratuit sur <https://urs.earthdata.nasa.gov>.
2. Ouvrez <https://urs.earthdata.nasa.gov/profile> → onglet **Generate Token** → copiez la chaîne.
3. Fournissez-la au notebook par l'un des quatre canaux ci-dessous.

| Environnement | Méthode recommandée |
|---|---|
| **Jupyter local** | un fichier `.env` à côté du notebook, contenant `EARTHDATA_TOKEN=votre_jeton` |
| **Google Colab** | panneau 🔑 *Secrets* → nouveau secret nommé `EARTHDATA_TOKEN`, accès activé pour ce notebook |
| **Kaggle** | *Add-ons → Secrets* → secret nommé `EARTHDATA_TOKEN` |
| **N'importe où** | saisie interactive masquée, proposée en dernier recours (le jeton n'est pas stocké) |


<div style="background:#FFF6E0;border:1px solid #D49A0044;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#D49A00;">⚠️ Ne collez jamais votre jeton dans une cellule</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">Un jeton écrit en clair part avec le notebook : dans le dépôt GitHub de l'atelier, dans la pièce jointe d'un courriel, dans l'historique des versions. Il donne accès à votre compte NASA. Les quatre canaux ci-dessus le gardent <b>en dehors</b> du fichier <code>.ipynb</code>. Si un jeton a fui, révoquez-le depuis la page profil Earthdata — c'est immédiat et gratuit. Pensez aussi à ajouter <code>.env</code> à votre <code>.gitignore</code>.</div></div>

In [ ]:
# =============================================================================
#  ÉTAPE 3 — Récupération du jeton Earthdata (sans jamais l'écrire ici)
# =============================================================================
import getpass

# Mettre à False pour ne jamais afficher d'invite de saisie (exécution automatisée)
DEMANDER_SI_ABSENT = True

# Noms acceptés, par ordre de préférence — compatibilité avec les notebooks 2024
NOMS_JETON = ["EARTHDATA_TOKEN", "EARTHDATA_BEARER", "LAADS_TOKEN"]


def lire_fichier_env(depart=None, remontees=3):
    """Lit un .env dans le dossier courant puis dans ses parents. Aucune dépendance."""
    depart = Path(depart or Path.cwd())
    candidats = [depart / ".env"] + [p / ".env" for p in list(depart.parents)[:remontees]]
    candidats.append(Path.home() / ".env")
    valeurs = {}
    for f in candidats:
        try:
            if not f.exists():
                continue
            for brut in f.read_text(encoding="utf-8").splitlines():
                brut = brut.strip()
                if not brut or brut.startswith("#") or "=" not in brut:
                    continue
                cle, val = brut.split("=", 1)
                valeurs.setdefault(cle.strip(), val.strip().strip('"').strip("'"))
            if valeurs:
                return valeurs, f
        except Exception:
            continue
    return {}, None


def _depuis_colab():
    try:
        from google.colab import userdata          # type: ignore
        for n in NOMS_JETON:
            try:
                v = userdata.get(n)
                if v:
                    return v.strip()
            except Exception:
                continue
    except Exception:
        pass
    return None


def _depuis_kaggle():
    try:
        from kaggle_secrets import UserSecretsClient   # type: ignore
        client = UserSecretsClient()
        for n in NOMS_JETON:
            try:
                v = client.get_secret(n)
                if v:
                    return v.strip()
            except Exception:
                continue
    except Exception:
        pass
    return None


def obtenir_jeton():
    """Parcourt les sources dans l'ordre et renvoie (jeton, origine)."""
    # 1) variables d'environnement du processus
    for n in NOMS_JETON:
        v = os.environ.get(n, "").strip()
        if v:
            return v, f"variable d'environnement {n}"

    # 2) fichier .env
    env, fichier = lire_fichier_env()
    for n in NOMS_JETON:
        if env.get(n):
            return env[n].strip(), f"fichier {fichier}"

    # 3) coffre de la plateforme
    if PLATEFORME == "Colab":
        v = _depuis_colab()
        if v:
            return v, "Secrets Google Colab"
    if PLATEFORME == "Kaggle":
        v = _depuis_kaggle()
        if v:
            return v, "Secrets Kaggle"

    # 4) saisie interactive masquée
    if DEMANDER_SI_ABSENT:
        try:
            v = getpass.getpass("Jeton Earthdata (saisie masquée, Entrée pour ignorer) : ").strip()
            if v:
                return v, "saisie interactive (non conservée)"
        except Exception:
            pass
    return "", "aucune"


JETON, ORIGINE_JETON = obtenir_jeton()

titre("Identifiants Earthdata")
if JETON:
    empreinte = f"{JETON[:6]}…{JETON[-4:]}" if len(JETON) > 14 else "(trop court ?)"
    ligne("Jeton détecté", f"{len(JETON)} caractères — {empreinte}", SYM["ok"])
    ligne("Origine", ORIGINE_JETON, SYM["pt"])
    if len(JETON) < 40 or " " in JETON:
        print(f"\n  {SYM['att']} Ce jeton paraît inhabituel (trop court, ou contenant un espace).")
        print("     Vérifiez qu'il a été copié en entier depuis urs.earthdata.nasa.gov/profile.")
else:
    ligne("Jeton détecté", "aucun", SYM["att"])
    print(f"\n  {SYM['att']} Les étapes 1 à 7 fonctionnent quand même : la recherche dans le catalogue")
    print("     NASA CMR est publique. Seule l'étape 8 (téléchargement) a besoin du jeton.")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 04 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Paramètres de la collecte</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">La seule cellule que vous avez à modifier</div></div>

Tout le comportement du notebook tient dans la cellule suivante, et le principe de la grille temporelle mérite d'être compris avant de la modifier.

### Le principe : une date fixe, plusieurs années

Comparer le 15 janvier 2015 au 3 août 2023 n'a pas de sens en lumières nocturnes. La radiance varie fortement avec la saison — végétation, nuages, angle solaire, neige aux hautes latitudes — et une bonne partie de l'écart que vous mesureriez serait de la saisonnalité, pas du développement. La parade est simple et standard : on fixe la date, et on ne fait varier que l'année.

Vous renseignez donc **trois choses** au lieu de quatre fenêtres :

| Paramètre | Ce qu'il fixe | Produits concernés |
|---|---|---|
| `ANNEES` | les années comparées | les quatre |
| `JOUR_FIXE` | le jour, au format `MM-JJ` | `VNP46A1`, `VNP46A2` (quotidiens) |
| `MOIS_FIXE` | le mois, de 1 à 12 | `VNP46A3` (mensuel) |

Avec les valeurs par défaut — `JOUR_FIXE = "01-15"`, `MOIS_FIXE = 1`, années 2013 à 2025 — le notebook va chercher les 15 janvier de treize années consécutives pour les produits quotidiens, les treize mois de janvier pour le mensuel, et les treize millésimes pour l'annuel. Vous obtenez quatre séries comparables, alignées sur la même saison.

### Choisir sa date de référence

Un bon choix de `JOUR_FIXE` évite la nouvelle Lune comme la pleine Lune, et une saison de forte nébulosité. Sous les tropiques, la **saison sèche** donne les nuits les plus dégagées : janvier pour l'Afrique de l'Ouest et le Sahel, juillet pour l'Afrique australe. Un jour unique reste exposé aux nuages : si la série a des trous, l'étape 7 vous les montrera année par année, et il suffira de décaler la date de quelques jours.

Le 29 février est géré : les années non bissextiles sont simplement signalées et ignorées.

In [ ]:
# =============================================================================
#  ÉTAPE 4 — PARAMÈTRES  ◄── la seule cellule à modifier
# =============================================================================

# --- 4.1  Le pays ------------------------------------------------------------
PAYS_ISO3 = "CIV"          # CIV, SDN, RWA, TUN, MAR, KEN, NGA, ZAF, EGY, SEN...

# Emprise personnalisée, si vous voulez une zone plus fine qu'un pays entier :
# (ouest, sud, est, nord) en degrés décimaux. Laisser None pour utiliser
# l'emprise nationale du répertoire intégré (étape 5).
BBOX_MANUELLE = None       # ex. (-4.30, 5.20, -3.70, 5.60)  -> le Grand Abidjan

MARGE_BBOX_DEG = 0.0       # marge ajoutée autour de l'emprise, en degrés


# --- 4.2  La grille temporelle : une date fixe, plusieurs années -------------
ANNEES    = list(range(2013, 2026))   # les années comparées
JOUR_FIXE = "01-15"                   # MM-JJ  -> pour VNP46A1 et VNP46A2
MOIS_FIXE = 1                         # 1..12  -> pour VNP46A3
                                      # VNP46A4 est annuel : rien à fixer

PRODUITS_A_TESTER = ["VNP46A1", "VNP46A2", "VNP46A3", "VNP46A4"]


# --- 4.3  Le téléchargement --------------------------------------------------
MODE_TELECHARGEMENT = "echantillon"      # "aucun" | "echantillon" | "complet"
PRODUITS_A_TELECHARGER = ["VNP46A4"]     # une liste : ajoutez-en autant que voulu
MAX_FICHIERS_PAR_PRODUIT = 2             # plafond en mode "echantillon"


# --- 4.4  Fiche d'identité des produits (documentation, pas un réglage) ------
CATALOGUE_PRODUITS = {
    "VNP46A1": dict(libelle="Radiance brute au capteur", frequence="quotidien",
                    depuis="2012-01-19", variable="DNB_At_Sensor_Radiance_500m",
                    doi="10.5067/VIIRS/VNP46A1.001",
                    usage="Diagnostic : voir la donnée avant toute correction."),
    "VNP46A2": dict(libelle="NTL corrigé BRDF lunaire, comblé", frequence="quotidien",
                    depuis="2012-01-19", variable="Gap_Filled_DNB_BRDF-Corrected_NTL",
                    doi="10.5067/VIIRS/VNP46A2.001",
                    usage="Suivi fin : crises, coupures, événements datés."),
    "VNP46A3": dict(libelle="Composite mensuel", frequence="mensuel",
                    depuis="2012-01-01", variable="NearNadir_Composite_Snow_Free",
                    doi="10.5067/VIIRS/VNP46A3.001",
                    usage="Saisonnalité et séries infra-annuelles."),
    "VNP46A4": dict(libelle="Composite annuel", frequence="annuel",
                    depuis="2012-01-01", variable="NearNadir_Composite_Snow_Free",
                    doi="10.5067/VIIRS/VNP46A4.001",
                    usage="Séries longues, comparaisons régionales, indicateurs."),
}

MOIS_FR = ["", "janvier", "février", "mars", "avril", "mai", "juin", "juillet",
           "août", "septembre", "octobre", "novembre", "décembre"]

# --- 4.5  Contrôles de cohérence immédiats -----------------------------------
assert MODE_TELECHARGEMENT in {"aucun", "echantillon", "complet"}, \
    "MODE_TELECHARGEMENT doit valoir 'aucun', 'echantillon' ou 'complet'."
assert set(PRODUITS_A_TESTER) <= set(CATALOGUE_PRODUITS), \
    f"Produits inconnus : {set(PRODUITS_A_TESTER) - set(CATALOGUE_PRODUITS)}"
assert set(PRODUITS_A_TELECHARGER) <= set(PRODUITS_A_TESTER) or MODE_TELECHARGEMENT == "aucun", \
    "Chaque produit de PRODUITS_A_TELECHARGER doit figurer dans PRODUITS_A_TESTER."
assert ANNEES, "ANNEES ne peut pas être vide."
assert min(ANNEES) >= 2012, "Black Marble commence en 2012 : aucune donnée avant."
assert 1 <= MOIS_FIXE <= 12, "MOIS_FIXE doit être compris entre 1 et 12."
try:
    _m, _j = (int(x) for x in JOUR_FIXE.split("-"))
    dt.date(2016, _m, _j)                     # 2016 est bissextile : valide le 29/02
except Exception:
    raise AssertionError("JOUR_FIXE doit être au format \"MM-JJ\", par exemple \"01-15\".")

titre("Paramètres retenus")
ligne("Pays (ISO3)", PAYS_ISO3, SYM["ok"])
ligne("Emprise", "personnalisée" if BBOX_MANUELLE else "nationale (répertoire intégré)", SYM["pt"])
ligne("Années comparées", f"{len(ANNEES)} — de {min(ANNEES)} à {max(ANNEES)}", SYM["ok"])
ligne("Jour fixe (quotidien)", f"{int(JOUR_FIXE.split('-')[1])} {MOIS_FR[int(JOUR_FIXE.split('-')[0])]}",
      SYM["pt"])
ligne("Mois fixe (mensuel)", MOIS_FR[MOIS_FIXE], SYM["pt"])
ligne("Produits testés", ", ".join(PRODUITS_A_TESTER), SYM["ok"])
ligne("Mode de téléchargement", MODE_TELECHARGEMENT, SYM["ok"])
if MODE_TELECHARGEMENT != "aucun":
    ligne("  produits visés", ", ".join(PRODUITS_A_TELECHARGER), SYM["pt"])
    ligne("  plafond par produit",
          "aucun (complet)" if MODE_TELECHARGEMENT == "complet"
          else f"{MAX_FICHIERS_PAR_PRODUIT} fichier(s)", SYM["pt"])

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 05 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Emprise nationale et tuiles VIIRS</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Du code ISO3 à la liste des tuiles, par le calcul</div></div>

Deux opérations s'enchaînent ici, et aucune ne demande le réseau.

**a) Du code ISO3 à une emprise.** Le notebook embarque un répertoire de 182 emprises nationales (rectangles englobants issus de Natural Earth, dont les 55 États membres de l'Union africaine). Pas de fichier externe, pas d'appel réseau : le notebook reste utilisable hors connexion et dans un environnement verrouillé. Si votre pays manque, ou si vous visez une zone plus fine qu'un pays, renseignez `BBOX_MANUELLE` à l'étape 4.

**b) De l'emprise aux tuiles.** La grille Black Marble est régulière, donc la conversion est une simple division entière — pas besoin d'interroger un serveur pour savoir quelles tuiles couvrent un pays :

$$h = \left\lfloor \frac{\lambda + 180}{10} \right\rfloor \qquad v = \left\lfloor \frac{90 - \varphi}{10} \right\rfloor$$

où $\lambda$ est la longitude et $\varphi$ la latitude. On applique la formule aux quatre coins de l'emprise, et le produit cartésien des indices obtenus donne le jeu de tuiles. La cellule vérifie ce calcul sur deux cas connus — la Côte d'Ivoire (`h17v07`, `h17v08`) et le Soudan (six tuiles) — avant de l'appliquer à votre pays.

In [ ]:
# =============================================================================
#  ÉTAPE 5 — Emprise du pays, puis tuiles VIIRS qui la recouvrent
# =============================================================================
import math, difflib

# --- 5.1  Répertoire intégré : ISO3|ISO2|nom|ouest|sud|est|nord|région -------
# Rectangles englobants nationaux (degrés décimaux, WGS84). Les 55 pays de
# l'Union africaine figurent en tête, le reste du monde ensuite.
_TABLE_EMPRISES = """
AGO|AO|Angola|11.64|-17.93|24.08|-4.44|AF
BFA|BF|Burkina Faso|-5.47|9.61|2.18|15.12|AF
BDI|BI|Burundi|29.02|-4.50|30.75|-2.35|AF
BEN|BJ|Bénin|0.77|6.14|3.80|12.24|AF
BWA|BW|Botswana|19.90|-26.83|29.43|-17.66|AF
COD|CD|République démocratique du Congo|12.18|-13.26|31.17|5.26|AF
CAF|CF|République centrafricaine|14.46|2.27|27.37|11.14|AF
COG|CG|République du Congo|11.09|-5.04|18.45|3.73|AF
CIV|CI|Côte d'Ivoire|-8.60|4.34|-2.56|10.52|AF
CMR|CM|Cameroun|8.49|1.73|16.01|12.86|AF
CPV|CV|Cap-Vert|-25.36|14.80|-22.66|17.21|AF
DJI|DJ|Djibouti|41.66|10.93|43.32|12.70|AF
DZA|DZ|Algérie|-8.68|19.06|12.00|37.12|AF
EGY|EG|Égypte|24.70|22.00|36.87|31.59|AF
ESH|EH|Sahara occidental|-17.10|20.77|-8.67|27.67|AF
ERI|ER|Érythrée|36.32|12.46|43.08|18.00|AF
ETH|ET|Éthiopie|32.95|3.42|47.79|14.96|AF
GAB|GA|Gabon|8.80|-3.98|14.43|2.33|AF
GHA|GH|Ghana|-3.24|4.71|1.06|11.10|AF
GMB|GM|Gambie|-16.84|13.13|-13.84|13.88|AF
GIN|GN|Guinée|-15.13|7.31|-7.83|12.59|AF
GNQ|GQ|Guinée Équatoriale|9.31|1.01|11.29|2.28|AF
GNB|GW|Guinée-Bissau|-16.68|11.04|-13.70|12.63|AF
KEN|KE|Kenya|33.89|-4.68|41.86|5.51|AF
COM|KM|Comores|43.21|-12.42|44.54|-11.36|AF
LBR|LR|Libéria|-11.44|4.36|-7.54|8.54|AF
LSO|LS|Lesotho|27.00|-30.65|29.33|-28.65|AF
LBY|LY|Libye|9.32|19.58|25.16|33.14|AF
MAR|MA|Maroc|-17.02|21.42|-1.12|35.76|AF
MDG|MG|Madagascar|43.25|-25.60|50.48|-12.04|AF
MLI|ML|Mali|-12.17|10.10|4.27|24.97|AF
MRT|MR|Mauritanie|-17.06|14.62|-4.92|27.40|AF
MUS|MU|Maurice|57.30|-20.55|63.51|-19.65|AF
MWI|MW|Malawi|32.69|-16.80|35.77|-9.23|AF
MOZ|MZ|Mozambique|30.18|-26.74|40.78|-10.32|AF
NAM|NA|Namibie|11.73|-29.05|25.08|-16.94|AF
NER|NE|Niger|0.30|11.66|15.90|23.47|AF
NGA|NG|Nigeria|2.69|4.24|14.58|13.87|AF
RWA|RW|Rwanda|29.02|-2.92|30.82|-1.13|AF
SYC|SC|Seychelles|55.17|-4.85|55.95|-3.69|AF
SDN|SD|Soudan|21.94|8.62|38.41|22.00|AF
SLE|SL|Sierra Leone|-13.25|6.79|-10.23|10.05|AF
SEN|SN|Sénégal|-17.63|12.33|-11.47|16.60|AF
SOM|SO|Somalie|40.98|-1.68|51.13|12.02|AF
SSD|SS|Soudan du Sud|23.89|3.51|35.30|12.25|AF
STP|ST|Sao Tomé-et-Principe|6.46|0.02|7.47|1.72|AF
SWZ|SZ|Eswatini|30.68|-27.29|32.07|-25.66|AF
TCD|TD|Tchad|13.54|7.42|23.89|23.41|AF
TGO|TG|Togo|-0.05|5.93|1.87|11.02|AF
TUN|TN|Tunisie|7.52|30.31|11.49|37.35|AF
TZA|TZ|Tanzanie, République unie de|29.34|-11.72|40.32|-0.95|AF
UGA|UG|Ouganda|29.58|-1.44|35.04|4.25|AF
ZAF|ZA|Afrique du Sud|16.34|-34.82|32.83|-22.09|AF
ZMB|ZM|Zambie|21.89|-17.96|33.49|-8.24|AF
ZWE|ZW|Zimbabwe|25.26|-22.27|32.85|-15.51|AF
ARE|AE|Émirats arabes unis|51.58|22.50|56.40|26.06|--
AFG|AF|Afghanistan|60.53|29.32|75.16|38.49|--
ALB|AL|Albanie|19.30|39.62|21.02|42.69|--
ARM|AM|Arménie|43.58|38.74|46.51|41.25|--
ATA|AQ|Antarctique|-180.00|-90.00|180.00|-63.27|--
ARG|AR|Argentine|-73.42|-55.25|-53.63|-21.83|--
AUT|AT|Autriche|9.48|46.43|16.98|49.04|--
AUS|AU|Australie|113.34|-43.63|153.57|-10.67|--
AZE|AZ|Azerbaïdjan|44.79|38.27|50.39|41.86|--
BIH|BA|Bosnie-Herzégovine|15.75|42.65|19.60|45.23|--
BGD|BD|Bangladesh|88.08|20.67|92.67|26.45|--
BEL|BE|Belgique|2.51|49.53|6.16|51.48|--
BGR|BG|Bulgarie|22.38|41.23|28.56|44.23|--
BHR|BH|Bahreïn|50.45|25.79|50.82|26.29|--
BRN|BN|Brunéi Darussalam|114.20|4.01|115.45|5.45|--
BOL|BO|Bolivie, état plurinational de|-69.59|-22.87|-57.50|-9.76|--
BRA|BR|Brésil|-73.99|-33.77|-34.73|5.24|--
BHS|BS|Bahamas|-78.98|23.71|-77.00|27.04|--
BTN|BT|Bhoutan|88.81|26.72|92.10|28.30|--
BLR|BY|Bélarus|23.20|51.32|32.69|56.17|--
BLZ|BZ|Belize|-89.23|15.89|-88.11|18.50|--
CAN|CA|Canada|-141.00|41.68|-52.65|73.23|--
CHE|CH|Suisse|6.02|45.78|10.44|47.83|--
CHL|CL|Chili|-75.64|-55.61|-66.96|-17.58|--
CHN|CN|Chine|73.68|18.20|135.03|53.46|--
COL|CO|Colombie|-78.99|-4.30|-66.88|12.44|--
CRI|CR|Costa Rica|-85.94|8.23|-82.55|11.22|--
CUB|CU|Cuba|-84.97|19.86|-74.18|23.19|--
CYP|CY|Chypre|32.26|34.57|34.00|35.17|--
CZE|CZ|Tchéquie|12.24|48.56|18.85|51.12|--
DEU|DE|Allemagne|5.99|47.30|15.02|54.98|--
DNK|DK|Danemark|8.09|54.80|12.69|57.73|--
DOM|DO|République dominicaine|-71.95|17.60|-68.32|19.88|--
ECU|EC|Équateur|-80.97|-4.96|-75.23|1.38|--
EST|EE|Estonie|23.34|57.47|28.13|59.61|--
ESP|ES|Espagne|-9.39|35.95|3.04|43.75|--
FIN|FI|Finlande|20.65|59.85|31.52|70.16|--
FJI|FJ|Fidji|-180.00|-18.29|180.00|-16.02|--
FLK|FK|Malouines, Îles (Falkland)|-61.20|-52.30|-57.75|-51.10|--
FRA|FR|France|-5.00|42.50|9.56|51.15|--
GBR|GB|Royaume-Uni|-7.57|49.96|1.68|58.64|--
GEO|GE|Géorgie|39.96|41.06|46.64|43.55|--
GRL|GL|Groënland|-73.30|60.04|-12.21|83.65|--
GRC|GR|Grèce|20.15|34.92|26.60|41.83|--
GTM|GT|Guatemala|-92.23|13.74|-88.23|17.82|--
GUY|GY|Guyana|-61.41|1.27|-56.54|8.37|--
HND|HN|Honduras|-89.35|12.98|-83.15|16.01|--
HRV|HR|Croatie|13.66|42.48|19.39|46.50|--
HTI|HT|Haïti|-74.46|18.03|-71.62|19.92|--
HUN|HU|Hongrie|16.20|45.76|22.71|48.62|--
IDN|ID|Indonésie|95.29|-10.36|141.03|5.48|--
IRL|IE|Irlande|-9.98|51.67|-6.03|55.13|--
ISR|IL|Israël|34.27|29.50|35.84|33.28|--
IND|IN|Inde|68.18|7.97|97.40|35.49|--
IRQ|IQ|Irak|38.79|29.10|48.57|37.39|--
IRN|IR|Iran, République islamique d'|44.11|25.08|63.32|39.71|--
ISL|IS|Islande|-24.33|63.50|-13.61|66.53|--
ITA|IT|Italie|6.75|36.62|18.48|47.12|--
JAM|JM|Jamaïque|-78.34|17.70|-76.20|18.52|--
JOR|JO|Jordanie|34.92|29.20|39.20|33.38|--
JPN|JP|Japon|129.41|31.03|145.54|45.55|--
KGZ|KG|Kirghizistan|69.46|39.28|80.26|43.30|--
KHM|KH|Cambodge|102.35|10.49|107.61|14.57|--
PRK|KP|Corée|124.27|37.67|130.78|42.99|--
KOR|KR|Corée, République de|126.12|34.39|129.47|38.61|--
KWT|KW|Koweït|46.57|28.53|48.42|30.06|--
KAZ|KZ|Kazakhstan|46.47|40.66|87.36|55.39|--
LAO|LA|Lao|100.12|13.88|107.56|22.46|--
LBN|LB|Liban|35.13|33.09|36.61|34.64|--
LKA|LK|Sri Lanka|79.70|5.97|81.79|9.82|--
LTU|LT|Lituanie|21.06|53.91|26.59|56.37|--
LUX|LU|Luxembourg|5.67|49.44|6.24|50.13|--
LVA|LV|Lettonie|21.06|55.62|28.18|57.97|--
MDA|MD|Moldova, République de|26.62|45.49|30.02|48.47|--
MNE|ME|Monténégro|18.45|41.88|20.34|43.52|--
MKD|MK|Macédoine du Nord|20.46|40.84|22.95|42.32|--
MMR|MM|Birmanie|92.30|9.93|101.18|28.34|--
MNG|MN|Mongolie|87.75|41.60|119.77|52.05|--
MLT|MT|Malte|14.18|35.79|14.58|36.08|--
MEX|MX|Mexique|-117.13|14.54|-86.81|32.72|--
MYS|MY|Malaisie|100.09|0.77|119.18|6.93|--
NCL|NC|Nouvelle-Calédonie|164.03|-22.40|167.12|-20.11|--
NIC|NI|Nicaragua|-87.67|10.73|-83.15|15.02|--
NLD|NL|Pays-Bas|3.31|50.80|7.09|53.51|--
NOR|NO|Norvège|4.99|58.08|31.29|70.92|--
NPL|NP|Népal|80.09|26.40|88.17|30.42|--
NZL|NZ|Nouvelle-Zélande|166.51|-46.64|178.52|-34.45|--
OMN|OM|Oman|52.00|16.65|59.81|26.40|--
PAN|PA|Panama|-82.97|7.22|-77.24|9.61|--
PER|PE|Pérou|-81.41|-18.35|-68.67|-0.06|--
PNG|PG|Papouasie-Nouvelle-Guinée|141.00|-10.65|156.02|-2.50|--
PHL|PH|Philippines|117.17|5.58|126.54|18.51|--
PAK|PK|Pakistan|60.87|23.69|77.84|37.13|--
POL|PL|Pologne|14.07|49.03|24.03|54.85|--
PRI|PR|Porto Rico|-67.24|17.95|-65.59|18.52|--
PSE|PS|Palestine, État de|34.93|31.35|35.55|32.53|--
PRT|PT|Portugal|-9.53|36.84|-6.39|42.28|--
PRY|PY|Paraguay|-62.69|-27.55|-54.29|-19.34|--
QAT|QA|Qatar|50.74|24.56|51.61|26.11|--
ROU|RO|Roumanie|20.22|43.69|29.63|48.22|--
SRB|RS|Serbie|18.83|42.25|22.99|46.17|--
RUS|RU|Russie, Fédération de|-180.00|41.15|180.00|81.25|--
SAU|SA|Arabie saoudite|34.63|16.35|55.67|32.16|--
SLB|SB|Salomon, Îles|156.49|-10.83|162.40|-6.60|--
SWE|SE|Suède|11.03|55.36|23.90|69.11|--
SGP|SG|Singapour|103.60|1.15|104.09|1.47|--
SVN|SI|Slovénie|13.70|45.45|16.56|46.85|--
SVK|SK|Slovaquie|16.88|47.76|22.56|49.57|--
SUR|SR|Surinam|-58.04|1.82|-53.96|6.03|--
SLV|SV|Salvador|-90.10|13.15|-87.72|14.42|--
SYR|SY|Syrienne, République arabe|35.70|32.31|42.35|37.23|--
ATF|TF|Terres australes françaises|68.72|-49.78|70.56|-48.63|--
THA|TH|Thaïlande|97.38|5.69|105.59|20.42|--
TJK|TJ|Tadjikistan|67.44|36.74|74.98|40.96|--
TLS|TL|Timor oriental|124.97|-9.39|127.34|-8.27|--
TKM|TM|Turkménistan|52.50|35.27|66.55|42.75|--
TUR|TR|Turquie|26.04|35.82|44.79|42.14|--
TTO|TT|Trinité-et-Tobago|-61.95|10.00|-60.90|10.89|--
TWN|TW|Taïwan, province de Chine|120.11|21.97|121.95|25.30|--
UKR|UA|Ukraine|22.09|44.36|40.08|52.34|--
USA|US|États-Unis|-125.00|25.00|-66.96|49.50|--
URY|UY|Uruguay|-58.43|-34.95|-53.21|-30.11|--
UZB|UZ|Ouzbékistan|55.93|37.14|73.06|45.59|--
VEN|VE|Vénézuela|-73.30|0.72|-59.76|12.16|--
VNM|VN|Viêt Nam|102.17|8.60|109.34|23.35|--
VUT|VU|Vanuatu|166.63|-16.60|167.84|-14.63|--
YEM|YE|Yémen|42.60|12.59|53.11|19.00|--
"""


def _charger_emprises():
    table = {}
    for brut in _TABLE_EMPRISES.strip().splitlines():
        iso3, iso2, nom, o, s, e, n, reg = brut.split("|")
        table[iso3] = dict(iso2=iso2, nom=nom, region=reg,
                           bbox=(float(o), float(s), float(e), float(n)))
    return table


EMPRISES = _charger_emprises()


def resoudre_pays(iso3, bbox_manuelle=None, marge=0.0):
    """Renvoie (nom, bbox) pour un code ISO3. Message d'aide explicite si inconnu."""
    iso3 = (iso3 or "").strip().upper()

    if iso3 not in EMPRISES and bbox_manuelle is None:
        proches = difflib.get_close_matches(iso3, EMPRISES.keys(), n=4, cutoff=0.5)
        indice = f" Codes proches : {', '.join(proches)}." if proches else ""
        raise KeyError(
            f"Code ISO3 « {iso3} » absent du répertoire.{indice}\n"
            "    → Utilisez un code alpha-3 (CIV, SDN, RWA…), ou renseignez "
            "BBOX_MANUELLE = (ouest, sud, est, nord) à l'étape 4."
        )

    nom = EMPRISES[iso3]["nom"] if iso3 in EMPRISES else f"zone personnalisée ({iso3})"
    o, s, e, n = bbox_manuelle if bbox_manuelle else EMPRISES[iso3]["bbox"]

    if marge:
        o, s, e, n = o - marge, s - marge, e + marge, n + marge
    # Bornes physiques du globe
    o, e = max(-180.0, o), min(180.0, e)
    s, n = max(-90.0, s), min(90.0, n)
    if not (o < e and s < n):
        raise ValueError(f"Emprise incohérente : {(o, s, e, n)} — attendu (ouest, sud, est, nord).")
    return nom, (round(o, 4), round(s, 4), round(e, 4), round(n, 4))


def tuiles_depuis_bbox(bbox):
    """Applique h = (lon+180)//10 et v = (90-lat)//10 aux coins de l'emprise."""
    o, s, e, n = bbox
    eps = 1e-9                       # évite une tuile parasite si un bord tombe pile sur 10°
    h0 = int(math.floor((o + 180.0) / 10.0))
    h1 = int(math.floor((e + 180.0 - eps) / 10.0))
    v0 = int(math.floor((90.0 - n) / 10.0))
    v1 = int(math.floor((90.0 - s - eps) / 10.0))
    h0, h1 = max(0, min(h0, 35)), max(0, min(h1, 35))
    v0, v1 = max(0, min(v0, 17)), max(0, min(v1, 17))
    return [f"h{h:02d}v{v:02d}" for v in range(v0, v1 + 1) for h in range(h0, h1 + 1)]


# --- 5.2  Autotest : la formule doit retrouver des jeux de tuiles connus ------
_attendus = {"CIV": {"h17v07", "h17v08"},
             "SDN": {"h20v06", "h21v06", "h20v07", "h21v07", "h20v08", "h21v08"}}
for _iso, _ref in _attendus.items():
    _obtenu = set(tuiles_depuis_bbox(resoudre_pays(_iso)[1]))
    assert _obtenu == _ref, f"Autotest tuiles en échec pour {_iso} : {_obtenu} != {_ref}"

# --- 5.3  Application au pays demandé ----------------------------------------
NOM_PAYS, BBOX = resoudre_pays(PAYS_ISO3, BBOX_MANUELLE, MARGE_BBOX_DEG)
TUILES = tuiles_depuis_bbox(BBOX)

titre(f"Zone d'étude — {NOM_PAYS}")
ligne("Code ISO3", PAYS_ISO3, SYM["ok"])
ligne("Emprise (O, S, E, N)", ", ".join(f"{v:+.2f}°" for v in BBOX), SYM["pt"])
ligne("Étendue", f"{BBOX[2]-BBOX[0]:.2f}° × {BBOX[3]-BBOX[1]:.2f}°", SYM["pt"])
ligne("Tuiles VIIRS", f"{len(TUILES)} — {', '.join(TUILES)}", SYM["ok"])
print(f"\n  {SYM['pt']} Autotest de la formule des tuiles : Côte d'Ivoire et Soudan {SYM['ok']}")
print(f"  {SYM['pt']} Chaque granule = 1 tuile × 1 date. Nombre de fichiers = périodes × {len(TUILES)} tuiles.")

### La fiche d'identité des tuiles

Le tableau produit ci-dessous est la **référence géographique** de votre collecte. Pour chaque tuile retenue, il donne son identifiant, ses quatre coins en degrés, la part de l'emprise nationale qu'elle porte, et — point le plus utile pour la suite — **la fenêtre de pixels** à découper dans la matrice 2 400 × 2 400 pour ne garder que votre pays.

| Colonne | Ce qu'elle contient |
|---|---|
| `tuile`, `h`, `v` | l'identifiant et ses deux indices de grille |
| `lon_min` … `lat_max` | les quatre coins de la tuile, en degrés décimaux |
| `part_emprise_%` | la part de l'emprise nationale que porte cette tuile (la somme fait 100 %) |
| `col_min`, `col_max`, `lig_min`, `lig_max` | la fenêtre de pixels à découper, sur 2 400 × 2 400 |
| `exemple_granule` | le nom exact qu'aura un fichier de cette tuile |

Chaque tuile couvre 10° × 10° en 2 400 × 2 400 pixels, soit **15 secondes d'arc** par pixel — environ 460 m à l'équateur. La conversion d'une coordonnée en pixel est directe : `colonne = (λ − lon_min) / (10/2400)` et `ligne = (lat_max − φ) / (10/2400)`, la ligne 0 étant tout en haut de la tuile.

In [ ]:
# =============================================================================
#  ÉTAPE 5 bis — Table de référence des tuiles retenues
# =============================================================================
PIXELS_PAR_TUILE = 2400                       # grille Black Marble : 2400 × 2400
DEGRES_PAR_PIXEL = 10.0 / PIXELS_PAR_TUILE    # 15 secondes d'arc (~460 m à l'équateur)


def reference_tuiles(bbox, tuiles, produit_exemple="VNP46A4", annee_exemple=2024):
    # Fiche d'identité de chaque tuile : emprise, part du pays, fenêtre de pixels.
    o, s, e, n = bbox
    aire_pays = (e - o) * (n - s)
    lignes = []

    for t in tuiles:
        h, v = int(t[1:3]), int(t[4:6])
        t_o, t_e = h * 10.0 - 180.0, (h + 1) * 10.0 - 180.0      # bornes de la tuile
        t_n, t_s = 90.0 - v * 10.0, 90.0 - (v + 1) * 10.0

        # Intersection entre la tuile et l'emprise nationale
        i_o, i_e = max(o, t_o), min(e, t_e)
        i_s, i_n = max(s, t_s), min(n, t_n)
        part = 100.0 * max(0.0, i_e - i_o) * max(0.0, i_n - i_s) / aire_pays if aire_pays else 0.0

        # Fenêtre de pixels à découper dans la matrice 2400 × 2400
        col_min = max(0, int((i_o - t_o) / DEGRES_PAR_PIXEL))
        col_max = min(PIXELS_PAR_TUILE, int(math.ceil((i_e - t_o) / DEGRES_PAR_PIXEL)))
        lig_min = max(0, int((t_n - i_n) / DEGRES_PAR_PIXEL))
        lig_max = min(PIXELS_PAR_TUILE, int(math.ceil((t_n - i_s) / DEGRES_PAR_PIXEL)))

        # Position relative de la tuile dans le jeu (utile quand il y en a plusieurs)
        vs = sorted({int(x[4:6]) for x in tuiles})
        hs = sorted({int(x[1:3]) for x in tuiles})
        ns = "nord" if v == vs[0] else ("sud" if v == vs[-1] else "centre")
        oe = "ouest" if h == hs[0] else ("est" if h == hs[-1] else "centre")
        position = ns if len(vs) > 1 else ""
        position = (position + "-" + oe).strip("-") if len(hs) > 1 else position

        lignes.append(dict(
            tuile=t, h=h, v=v,
            lon_min=round(t_o, 2), lon_max=round(t_e, 2),
            lat_min=round(t_s, 2), lat_max=round(t_n, 2),
            position=position or "unique",
            part_emprise_pct=round(part, 1),
            col_min=col_min, col_max=col_max, lig_min=lig_min, lig_max=lig_max,
            pixels_utiles=(col_max - col_min) * (lig_max - lig_min),
            exemple_granule=f"{produit_exemple}.A{annee_exemple}001.{t}.001.<horodatage>.h5",
        ))
    return pd.DataFrame(lignes)


TUILES_REFERENCE = reference_tuiles(BBOX, TUILES)

titre(f"Référence des tuiles — {NOM_PAYS} ({PAYS_ISO3})")

for _, r in TUILES_REFERENCE.iterrows():
    print(f"\n  {SYM['ok']} {r['tuile']}   (h = {r['h']:02d}, v = {r['v']:02d})"
          f"   position : {r['position']}")
    print(f"       emprise          longitude {r['lon_min']:+7.2f}° {SYM['fl']} {r['lon_max']:+7.2f}°"
          f"   |   latitude {r['lat_min']:+6.2f}° {SYM['fl']} {r['lat_max']:+6.2f}°")
    print(f"       part de l'emprise nationale portée par cette tuile : {r['part_emprise_pct']:.1f} %")
    print(f"       fenêtre à découper   colonnes {r['col_min']:4d} {SYM['fl']} {r['col_max']:4d}"
          f"   |   lignes {r['lig_min']:4d} {SYM['fl']} {r['lig_max']:4d}"
          f"   ({r['pixels_utiles']/1e6:.2f} M pixels sur 5,76 M)")
    print(f"       nom de fichier       {r['exemple_granule']}")

print(f"\n  {SYM['pt']} Grille : {PIXELS_PAR_TUILE} × {PIXELS_PAR_TUILE} pixels par tuile, "
      f"{DEGRES_PAR_PIXEL*3600:.0f}\" d'arc par pixel (~460 m à l'équateur)")
print(f"  {SYM['pt']} Total des parts : {TUILES_REFERENCE['part_emprise_pct'].sum():.1f} % "
      f"(doit valoir 100 % — l'emprise est entièrement couverte)")

_ft = DOSSIERS["catalogue"] / f"tuiles_{PAYS_ISO3}.csv"
TUILES_REFERENCE.to_csv(_ft, index=False, encoding="utf-8-sig")
print(f"\n  {SYM['ok']} Table de référence écrite : {_ft}")
print(TUILES_REFERENCE[["tuile", "lon_min", "lon_max", "lat_min", "lat_max",
                        "part_emprise_pct", "col_min", "col_max",
                        "lig_min", "lig_max"]].to_string(index=False))

La figure ci-dessous situe le pays dans la grille Black Marble. Elle a une vraie utilité de contrôle : si le rectangle doré n'entoure pas le pays que vous visez, c'est que le code ISO3 ou l'emprise manuelle est erroné — autant le voir maintenant plutôt qu'après avoir téléchargé les mauvaises tuiles.

In [ ]:
# =============================================================================
#  ÉTAPE 5 ter — Contrôle visuel : le pays dans la grille de tuiles
# =============================================================================
if MPL_DISPO:
    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    o, s, e, n = BBOX
    h_idx = sorted({int(t[1:3]) for t in TUILES})
    v_idx = sorted({int(t[4:6]) for t in TUILES})
    h_vue = range(max(0, h_idx[0] - 1), min(35, h_idx[-1] + 1) + 1)
    v_vue = range(max(0, v_idx[0] - 1), min(17, v_idx[-1] + 1) + 1)

    largeur = min(11.0, max(5.4, 1.95 * len(list(h_vue)) + 2.0))
    hauteur = min(9.0, max(4.2, 1.95 * len(list(v_vue)) + 1.3))
    fig, ax = plt.subplots(figsize=(largeur, hauteur), dpi=110)
    for v in v_vue:
        for h in h_vue:
            x, y = h * 10 - 180, 90 - (v + 1) * 10
            retenue = f"h{h:02d}v{v:02d}" in TUILES
            ax.add_patch(Rectangle((x, y), 10, 10,
                                   facecolor="#E8F5EF" if retenue else "#FFFFFF",
                                   edgecolor="#00A86A" if retenue else "#D5DED9",
                                   linewidth=1.6 if retenue else 0.8, zorder=1))
            ax.text(x + 5, y + 5.6 if retenue else y + 5, f"h{h:02d}v{v:02d}",
                    ha="center", va="center", fontsize=8.5,
                    color="#00704A" if retenue else "#B6C2BC",
                    fontweight="bold" if retenue else "normal", zorder=2)
            if retenue:                       # part de l'emprise portée par la tuile
                part = float(TUILES_REFERENCE.loc[
                    TUILES_REFERENCE["tuile"] == f"h{h:02d}v{v:02d}", "part_emprise_pct"].iloc[0])
                ax.text(x + 5, y + 4.0, f"{part:.1f} %", ha="center",
                        va="center", fontsize=8.2, color="#5E6964", zorder=2)

    ax.add_patch(Rectangle((o, s), e - o, n - s, facecolor="#F5C24233",
                           edgecolor="#D49A00", linewidth=2.2, zorder=3))
    ax.set_xlim(min(h_vue) * 10 - 180, (max(h_vue) + 1) * 10 - 180)
    ax.set_ylim(90 - (max(v_vue) + 1) * 10, 90 - min(v_vue) * 10)
    ax.set_aspect("equal")
    ax.set_xlabel("Longitude (°)", fontsize=9, color="#5E6964")
    ax.set_ylabel("Latitude (°)", fontsize=9, color="#5E6964")
    ax.tick_params(labelsize=8, colors="#5E6964")
    for bord in ax.spines.values():
        bord.set_color("#D5DED9")
    ax.set_title(f"{NOM_PAYS} ({PAYS_ISO3}) — {len(TUILES)} tuile(s) de 10°×10°\n"
                 f"le pourcentage indique la part de l'emprise portée par chaque tuile",
                 fontsize=11.5, fontweight="bold", color="#231F20", pad=12)
    fig.tight_layout()
    _fig = DOSSIERS["figures"] / f"tuiles_{PAYS_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure enregistrée : {_fig}")
else:
    print(f"  {SYM['att']} matplotlib absent — contrôle visuel ignoré.")
    print(f"     Emprise {BBOX}  {SYM['fl']}  tuiles {TUILES}")

### La même chose sur une vraie carte

La grille abstraite ci-dessus dit *où sont les tuiles*. La carte interactive ci-dessous dit *ce qu'il y a dedans* : elle pose les mêmes rectangles sur un fond géographique réel, avec les frontières, les villes et le relief. C'est le contrôle le plus parlant — on voit immédiatement si l'emprise correspond au pays visé, et à quel point une tuile de 10° déborde d'un petit pays.

**Cliquez sur une tuile** : sa fiche de référence s'ouvre — coins en degrés, part de l'emprise, fenêtre de pixels, nom de fichier attendu. Le sélecteur en haut à droite bascule entre un fond clair, pour lire les frontières, et un fond sombre, plus naturel pour des lumières nocturnes.

La carte est aussi enregistrée en **HTML autonome** dans `figures/`. Ce fichier s'ouvre dans n'importe quel navigateur, s'envoie par courriel et se publie tel quel sur GitHub Pages — ce sera utile vendredi.

In [ ]:
# =============================================================================
#  ÉTAPE 5 quater — Carte interactive : les tuiles sur un fond géographique
# =============================================================================
def assurer_folium():
    # folium est une surcouche Python de Leaflet : pur Python, aucune dépendance
    # géospatiale compilée. Présent d'office sur Colab et Kaggle.
    try:
        import folium
        return folium
    except ModuleNotFoundError:
        print(f"  {SYM['fl']} Installation de folium ...")
        if installer(["folium"]):
            try:
                importlib.invalidate_caches()
                import folium
                return folium
            except Exception:
                pass
    return None


folium = assurer_folium()

if folium is None:
    print(f"  {SYM['att']} folium indisponible — la grille statique ci-dessus reste valable.")
else:
    b_o, b_s, b_e, b_n = BBOX
    carte = folium.Map(location=[(b_s + b_n) / 2, (b_o + b_e) / 2],
                       zoom_start=5, tiles=None, control_scale=True)
    folium.TileLayer('OpenStreetMap', name='Fond clair').add_to(carte)
    folium.TileLayer('CartoDB dark_matter', name='Fond sombre').add_to(carte)

    # --- Les tuiles Black Marble, une par une --------------------------------
    groupe = folium.FeatureGroup(name=f'Tuiles VIIRS ({len(TUILES)})', show=True)
    part_max = max(1e-9, float(TUILES_REFERENCE['part_emprise_pct'].max()))

    for _, r in TUILES_REFERENCE.iterrows():
        fiche = (
            '<div style="font-family:Segoe UI,Calibri,sans-serif;font-size:12.5px;min-width:290px;">'
            '<div style="background:#00553A;color:#fff;padding:7px 11px;border-radius:7px 7px 0 0;'
            'font-weight:700;font-size:14px;">'
            f'{r["tuile"]}'
            '<span style="color:#F5C242;font-weight:400;font-size:11.5px;">'
            f' &middot; h={r["h"]:02d} v={r["v"]:02d} &middot; {r["position"]}</span></div>'
            '<table style="border-collapse:collapse;width:100%;">'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Longitude</td>'
            f'<td style="padding:4px 10px;"><b>{r["lon_min"]:+.2f}&deg; &rarr; {r["lon_max"]:+.2f}&deg;</b></td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Latitude</td>'
            f'<td style="padding:4px 10px;"><b>{r["lat_min"]:+.2f}&deg; &rarr; {r["lat_max"]:+.2f}&deg;</b></td></tr>'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Part de l&rsquo;emprise</td>'
            f'<td style="padding:4px 10px;color:#00704A;"><b>{r["part_emprise_pct"]:.1f} %</b></td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Colonnes &agrave; d&eacute;couper</td>'
            f'<td style="padding:4px 10px;"><b>{r["col_min"]} &rarr; {r["col_max"]}</b> / 2400</td></tr>'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Lignes &agrave; d&eacute;couper</td>'
            f'<td style="padding:4px 10px;"><b>{r["lig_min"]} &rarr; {r["lig_max"]}</b> / 2400</td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Pixels utiles</td>'
            f'<td style="padding:4px 10px;"><b>{r["pixels_utiles"]/1e6:.2f} M</b> sur 5,76 M</td></tr>'
            '</table>'
            '<div style="padding:7px 10px;background:#E8F5EF;border-radius:0 0 7px 7px;'
            'font-family:Consolas,monospace;font-size:10.5px;word-break:break-all;">'
            f'{r["exemple_granule"]}</div></div>'
        )

        # Intensité du vert proportionnelle à la part de l'emprise portée
        poids = float(r['part_emprise_pct']) / part_max
        folium.Rectangle(
            bounds=[[r['lat_min'], r['lon_min']], [r['lat_max'], r['lon_max']]],
            color='#00A86A', weight=2.2, fill=True, fill_color='#00A86A',
            fill_opacity=0.06 + 0.20 * poids,
            tooltip=f'{r["tuile"]} : {r["part_emprise_pct"]:.1f} % de l emprise',
            popup=folium.Popup(fiche, max_width=340),
        ).add_to(groupe)

        etiquette = (
            '<div style="font-family:Segoe UI,Calibri,sans-serif;text-align:center;'
            'white-space:nowrap;transform:translate(-50%,-50%);">'
            '<div style="color:#00553A;font-weight:800;font-size:15px;'
            'text-shadow:0 0 4px #fff,0 0 9px #fff;">'
            f'{r["tuile"]}</div>'
            '<div style="color:#00704A;font-size:11.5px;'
            'text-shadow:0 0 4px #fff,0 0 9px #fff;">'
            f'{r["part_emprise_pct"]:.1f} %</div></div>'
        )
        folium.Marker(
            location=[(r['lat_min'] + r['lat_max']) / 2, (r['lon_min'] + r['lon_max']) / 2],
            icon=folium.DivIcon(html=etiquette),
        ).add_to(groupe)
    groupe.add_to(carte)

    # --- L'emprise nationale interrogée --------------------------------------
    emprise = folium.FeatureGroup(name='Emprise interrogée', show=True)
    folium.Rectangle(
        bounds=[[b_s, b_o], [b_n, b_e]], color='#D49A00', weight=3, dash_array='9,6',
        fill=True, fill_color='#F5C242', fill_opacity=0.10,
        tooltip=f'Emprise interrogée : {NOM_PAYS} ({PAYS_ISO3})',
        popup=folium.Popup(
            f'<b>{NOM_PAYS} ({PAYS_ISO3})</b><br>'
            f'Ouest {b_o:+.2f}&deg; &middot; Sud {b_s:+.2f}&deg;<br>'
            f'Est {b_e:+.2f}&deg; &middot; Nord {b_n:+.2f}&deg;<br>'
            f'&Eacute;tendue {b_e-b_o:.2f}&deg; &times; {b_n-b_s:.2f}&deg;', max_width=260),
    ).add_to(emprise)
    emprise.add_to(carte)

    # --- Légende flottante ---------------------------------------------------
    legende = (
        '<div style="position:fixed;bottom:22px;left:14px;z-index:9999;background:#fff;'
        'border:1px solid #D5DED9;border-radius:10px;padding:11px 15px;'
        'font-family:Segoe UI,Calibri,sans-serif;font-size:11.5px;color:#231F20;'
        'box-shadow:0 2px 10px #00000022;">'
        '<div style="color:#00704A;font-weight:800;letter-spacing:1.5px;font-size:10px;">'
        'NASA BLACK MARBLE &middot; VNP46</div>'
        '<div style="font-weight:700;margin:3px 0 7px 0;font-size:13px;">'
        f'{NOM_PAYS} ({PAYS_ISO3})</div>'
        '<div><span style="display:inline-block;width:13px;height:13px;background:#00A86A44;'
        'border:2px solid #00A86A;vertical-align:-2px;"></span>&nbsp;'
        f'{len(TUILES)} tuile(s) de 10&deg;&times;10&deg;</div>'
        '<div style="margin-top:4px;"><span style="display:inline-block;width:13px;height:13px;'
        'background:#F5C24233;border:2px dashed #D49A00;vertical-align:-2px;"></span>'
        '&nbsp;emprise interrog&eacute;e</div>'
        '<div style="color:#5E6964;margin-top:7px;font-size:10.5px;">'
        'Cliquez une tuile pour sa fiche de r&eacute;f&eacute;rence</div></div>'
    )
    carte.get_root().html.add_child(folium.Element(legende))
    folium.LayerControl(collapsed=False).add_to(carte)
    carte.fit_bounds([[b_s - 1, b_o - 1], [b_n + 1, b_e + 1]])

    _carte = DOSSIERS['figures'] / f'carte_tuiles_{PAYS_ISO3}.html'
    carte.save(str(_carte))
    print(f"  {SYM['ok']} Carte interactive : {_carte}")
    print(f"  {SYM['pt']} HTML autonome : ouvrable hors Jupyter, publiable sur GitHub Pages")

    try:
        from IPython.display import display
        display(carte)
    except Exception:
        print(f"  {SYM['att']} Affichage en ligne indisponible : ouvrez le fichier HTML.")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 06 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Interroger le catalogue NASA CMR</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Une fenêtre étroite par année, paginée et dédoublonnée</div></div>

Le point d'entrée est unique : `https://cmr.earthdata.nasa.gov/search/granules.json`. On lui transmet le nom court du produit, une fenêtre temporelle et une emprise ; il répond en JSON, avec pour chaque granule son nom exact, sa taille et ses liens de téléchargement.

### Une requête par année, et non une grande requête

La grille « date fixe × années » se traduit en **une fenêtre étroite par année**. Pour treize années de `VNP46A2`, le notebook envoie treize requêtes qui demandent chacune une seule journée, plutôt qu'une requête couvrant treize ans dont il faudrait ensuite jeter 99,98 % du résultat. C'est plus rapide, cela ne charge pas inutilement le serveur de la NASA, et surtout cela rend le diagnostic possible : si une année manque, on sait immédiatement laquelle.

La fenêtre dépend de la fréquence du produit :

| Produit | Fenêtre demandée pour l'année Y | Étiquette |
|---|---|---|
| `VNP46A1`, `VNP46A2` | la seule journée `Y-MM-JJ` | `2019-01-15` |
| `VNP46A3` | du 1er au dernier jour du mois fixé | `2019-01` |
| `VNP46A4` | l'année entière | `2019` |

### Trois précautions qui font la différence

**La pagination.** Le CMR plafonne à 2 000 résultats par page. Les fenêtres sont étroites ici, mais la boucle est là pour le jour où vous élargirez — sans elle, on perdrait une partie du catalogue en silence.

**Le dédoublonnage.** Un même granule peut être publié en plusieurs versions de collection (`.001`, `.002`). On conserve une seule entrée par couple *(tuile, date)*, en retenant la version la plus élevée.

**Le choix du lien.** Parmi les liens renvoyés figurent des URL OPeNDAP et S3, qui ne se téléchargent pas comme un fichier ordinaire. On filtre pour ne garder que le lien HTTPS direct vers le `.h5`.

In [ ]:
# =============================================================================
#  ÉTAPE 6 — Fenêtres temporelles et interrogation du CMR (aucune requête ici)
# =============================================================================
import re, time, json

CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"
CLIENT_ID = "BAD-GTS17-NTL"          # courtoisie : identifie l'appelant auprès du CMR
TAILLE_PAGE = 2000                   # maximum autorisé par le CMR
PAGES_MAX = 25                       # garde-fou : 50 000 granules au plus

MOTIF_GRANULE = re.compile(r"(VNP46A\d)\.A(\d{4})(\d{3})\.(h\d{2}v\d{2})\.(\d{3})\.", re.IGNORECASE)

SESSION_CMR = requests.Session()
SESSION_CMR.headers.update({"Client-Id": CLIENT_ID, "User-Agent": f"{CLIENT_ID}/1.0"})


# --- 6.1  La grille temporelle : date fixe × années --------------------------
def fenetre_annee(produit, annee):
    """Fenêtre à demander pour ce produit et cette année.

    Renvoie (étiquette, date de début, date de fin), ou None si la date
    n'existe pas cette année-là (le 29 février hors année bissextile).
    """
    freq = CATALOGUE_PRODUITS[produit]["frequence"]

    if freq == "quotidien":
        mois, jour = (int(x) for x in JOUR_FIXE.split("-"))
        try:
            d = dt.date(annee, mois, jour)
        except ValueError:
            return None                       # 29 février, année non bissextile
        return (d.isoformat(), d, d)

    if freq == "mensuel":
        d0 = dt.date(annee, MOIS_FIXE, 1)
        d1 = (dt.date(annee + (MOIS_FIXE == 12), MOIS_FIXE % 12 + 1, 1)
              - dt.timedelta(days=1))
        return (f"{annee}-{MOIS_FIXE:02d}", d0, d1)

    return (str(annee), dt.date(annee, 1, 1), dt.date(annee, 12, 31))


def fenetres(produit, annees):
    """Toutes les fenêtres valides d'un produit, dans l'ordre chronologique."""
    return [f for f in (fenetre_annee(produit, a) for a in annees) if f]


# --- 6.2  Lecture d'une réponse CMR ------------------------------------------
def _analyser_nom(fichier):
    """VNP46A4.A2023001.h17v08.001.2024031102344.h5 → dict ou None."""
    m = MOTIF_GRANULE.search(fichier or "")
    if not m:
        return None
    produit, annee, jour, tuile, version = m.groups()
    date = dt.date(int(annee), 1, 1) + dt.timedelta(days=int(jour) - 1)
    return dict(produit=produit.upper(), annee=int(annee), jour_julien=int(jour),
                tuile=tuile.lower(), version=int(version), date=date)


def _choisir_url(liens, fichier):
    """Garde le lien HTTPS direct vers le .h5 ; écarte OPeNDAP, S3 et les vignettes."""
    meilleurs = []
    for lien in liens or []:
        href = (lien.get("href") or "").strip()
        bas = href.lower()
        if not bas.startswith("http") or ".h5" not in bas:
            continue
        if bas.split("?")[0].rsplit("/", 1)[-1] != fichier.lower():
            continue
        score = 0
        if "opendap" in bas:
            score += 10                       # utilisable, mais pas pour un téléchargement simple
        if "s3credentials" in bas or bas.startswith("s3://"):
            score += 20
        if "/archive/" in bas or "ladsweb" in bas:
            score -= 5                        # le chemin d'archive classique, à privilégier
        meilleurs.append((score, href))
    return min(meilleurs)[1] if meilleurs else None


# --- 6.3  Une requête, une fenêtre -------------------------------------------
def _requete_cmr(produit, debut, fin, bbox, tuiles=None):
    """Interroge le CMR sur une fenêtre et renvoie une liste de granules."""
    parametres = {
        "short_name": produit,
        "temporal[]": f"{debut}T00:00:00Z,{fin}T23:59:59Z",
        "bounding_box[]": ",".join(f"{v}" for v in bbox),
        "page_size": TAILLE_PAGE,
        "sort_key": "start_date",
    }
    lignes, page = [], 1

    while page <= PAGES_MAX:
        parametres["page_num"] = page
        reponse = None
        for tentative in range(1, 4):                     # 3 essais, attente croissante
            try:
                reponse = SESSION_CMR.get(CMR_URL, params=parametres, timeout=90)
                reponse.raise_for_status()
                break
            except Exception as err:
                if tentative == 3:
                    raise RuntimeError(
                        f"Le catalogue CMR est injoignable pour {produit} ({err}).\n"
                        "    → Sur Kaggle, activez « Internet » dans le panneau de droite.\n"
                        "    → Derrière un proxy d'entreprise, exportez HTTPS_PROXY avant de lancer Jupyter."
                    ) from err
                time.sleep(3 * tentative)

        entrees = reponse.json().get("feed", {}).get("entry", [])
        for e in entrees:
            fichier = e.get("producer_granule_id") or e.get("title") or ""
            if not fichier.lower().endswith(".h5"):
                for lien in e.get("links", []):
                    base = (lien.get("href") or "").split("?")[0].rsplit("/", 1)[-1]
                    if base.lower().endswith(".h5"):
                        fichier = base
                        break
            infos = _analyser_nom(fichier)
            if not infos:
                continue
            if tuiles and infos["tuile"] not in {t.lower() for t in tuiles}:
                continue                                   # tuile hors de notre pays
            url = _choisir_url(e.get("links"), fichier)
            if not url:
                continue
            try:
                taille = float(e.get("granule_size") or 0.0)
            except (TypeError, ValueError):
                taille = 0.0
            lignes.append(dict(fichier=fichier, url=url, taille_mo=round(taille, 2), **infos))

        if len(entrees) < TAILLE_PAGE:                     # dernière page atteinte
            break
        page += 1
    return lignes


COLONNES_CATALOGUE = ["produit", "periode", "date", "annee", "jour_julien",
                      "tuile", "version", "taille_mo", "fichier", "url"]


def interroger_cmr(produit, annees, bbox, tuiles=None, trace=None):
    """Parcourt les fenêtres du produit, une année après l'autre."""
    lignes = []
    for etiquette, d0, d1 in fenetres(produit, annees):
        trouves = _requete_cmr(produit, d0, d1, bbox, tuiles)
        for g in trouves:
            g["periode"] = etiquette
        lignes.extend(trouves)
        if trace is not None:
            trace.append(dict(periode=etiquette, granules=len(trouves),
                              tuiles=len({g["tuile"] for g in trouves})))

    if not lignes:
        return pd.DataFrame(columns=COLONNES_CATALOGUE)

    df = pd.DataFrame(lignes)
    # Dédoublonnage : une seule entrée par (tuile, date), version la plus élevée
    df = (df.sort_values(["tuile", "date", "version"], ascending=[True, True, False])
            .drop_duplicates(subset=["produit", "tuile", "date"], keep="first")
            .sort_values(["date", "tuile"])
            .reset_index(drop=True))
    return df[COLONNES_CATALOGUE]


# --- 6.4  Aperçu de la grille, avant toute requête ---------------------------
titre("Grille temporelle demandée")
for p in PRODUITS_A_TESTER:
    f = fenetres(p, ANNEES)
    apercu = ", ".join(e for e, _, _ in f[:4])
    suite = f" … {f[-1][0]}" if len(f) > 4 else ""
    ligne(f"{p} ({CATALOGUE_PRODUITS[p]['frequence']})",
          f"{len(f)} fenêtre(s) : {apercu}{suite}", SYM["ok"])
    ignorees = len(ANNEES) - len(f)
    if ignorees:
        print(f"      {SYM['att']} {ignorees} année(s) sans le {JOUR_FIXE} "
              f"(29 février hors année bissextile)")

print(f"\n  {SYM['pt']} Total : {sum(len(fenetres(p, ANNEES)) for p in PRODUITS_A_TESTER)} "
      f"requêtes au catalogue, soit {len(TUILES)} tuile(s) attendue(s) par fenêtre")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 07 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Test de disponibilité des quatre produits</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Ce que la NASA publie réellement, année par année</div></div>

C'est le cœur du notebook. Pour chaque produit, le notebook parcourt ses fenêtres annuelles et compare trois grandeurs :

- **attendu** = nombre de fenêtres × nombre de tuiles, ce que la théorie prédit ;
- **trouvé** = ce que le catalogue renvoie réellement ;
- **complétude** = le rapport des deux.

Un écart n'est presque jamais un bug, et c'est toute la valeur de l'exercice. Le millésime annuel de l'année en cours n'est pas encore publié. Les produits quotidiens démarrent le 19 janvier 2012. Et surtout, sur un jour unique, une année peut manquer parce que la nuit était couverte — information capitale, qu'un téléchargement aveugle aurait masquée.

Le tableau donne aussi le **volume total** par produit, l'information décisive avant de lancer quoi que ce soit. Aucun jeton n'est nécessaire à cette étape.

In [ ]:
# =============================================================================
#  ÉTAPE 7 — Sonder les produits : disponibilité, complétude, volume
# =============================================================================
titre(f"Disponibilité Black Marble — {NOM_PAYS} ({PAYS_ISO3})")

inventaire, catalogue_complet, traces = [], [], {}
reseau_hors_service = False          # évite de réessayer 4 fois si le réseau est coupé

for produit in PRODUITS_A_TESTER:
    grille = fenetres(produit, ANNEES)
    attendu = len(grille) * len(TUILES)
    chrono = time.perf_counter()
    print(f"\n  {SYM['fl']} {produit} — {CATALOGUE_PRODUITS[produit]['libelle']}  "
          f"({len(grille)} fenêtre(s) : {grille[0][0]} → {grille[-1][0]})")

    trace = []
    if reseau_hors_service:
        df, erreur = pd.DataFrame(), "réseau indisponible (voir le produit précédent)"
        print(f"     {SYM['ko']} {erreur}")
    else:
        try:
            df = interroger_cmr(produit, ANNEES, BBOX, TUILES, trace=trace)
            erreur = None
        except Exception as exc:
            df, erreur = pd.DataFrame(), str(exc)
            reseau_hors_service = True
            for l in erreur.splitlines():
                print(f"     {SYM['ko']} {l.strip()}")

    duree = time.perf_counter() - chrono
    trouve = len(df)
    volume = float(df["taille_mo"].sum()) if trouve else 0.0
    periodes_ok = sorted(df["periode"].unique()) if trouve else []
    traces[produit] = trace

    inventaire.append(dict(
        produit=produit,
        frequence=CATALOGUE_PRODUITS[produit]["frequence"],
        grille=f"{grille[0][0]} → {grille[-1][0]}",
        fenetres=len(grille), tuiles=len(TUILES),
        attendu=attendu, trouve=trouve,
        complet_pct=round(100 * trouve / attendu, 1) if attendu else 0.0,
        annees_publiees=len(periodes_ok),
        volume_mo=round(volume, 1),
        mo_par_fichier=round(volume / trouve, 1) if trouve else 0.0,
        secondes=round(duree, 1),
        erreur=erreur or "",
    ))

    if trouve:
        catalogue_complet.append(df)
        marque = SYM["ok"] if trouve >= attendu else SYM["att"]
        print(f"     {marque} {trouve} granule(s) sur {attendu} attendu(s) — "
              f"{len(periodes_ok)}/{len(grille)} période(s) publiée(s) — "
              f"{volume/1024:.2f} Go — {duree:.1f} s")
    elif not erreur:
        print(f"     {SYM['att']} aucun granule trouvé sur cette grille.")

INVENTAIRE = pd.DataFrame(inventaire)
CATALOGUE = (pd.concat(catalogue_complet, ignore_index=True)
             if catalogue_complet else pd.DataFrame(columns=COLONNES_CATALOGUE))

# --- Tableau de synthèse -----------------------------------------------------
titre("Synthèse")
_vue = INVENTAIRE[["produit", "frequence", "grille", "fenetres", "tuiles", "attendu",
                   "trouve", "complet_pct", "annees_publiees", "volume_mo"]]
_vue = _vue.rename(columns={"complet_pct": "complet_%", "volume_mo": "volume_Mo",
                            "annees_publiees": "periodes_ok"})
print(_vue.to_string(index=False))

# --- Exports -----------------------------------------------------------------
_f1 = DOSSIERS["catalogue"] / f"inventaire_{PAYS_ISO3}.csv"
INVENTAIRE.to_csv(_f1, index=False, encoding="utf-8-sig")
print(f"\n  {SYM['ok']} Inventaire écrit  : {_f1}")
if len(CATALOGUE):
    _f2 = DOSSIERS["catalogue"] / f"granules_{PAYS_ISO3}.csv"
    CATALOGUE.to_csv(_f2, index=False, encoding="utf-8-sig")
    print(f"  {SYM['ok']} Catalogue écrit   : {_f2}  ({len(CATALOGUE)} granules)")
    print(f"  {SYM['pt']} Volume total si tout était téléchargé : "
          f"{CATALOGUE['taille_mo'].sum()/1024:.2f} Go")

Le détail ci-dessous répond à la question qui suit immédiatement : **quelles années manquent, exactement, et pour quel produit ?** Une année incomplète — moins de tuiles que prévu — est signalée séparément d'une année totalement absente : la première se corrige en décalant la date de quelques jours, la seconde révèle souvent une limite du produit lui-même.

In [ ]:
# =============================================================================
#  ÉTAPE 7 bis — Couverture période par période
# =============================================================================
if len(CATALOGUE):
    for produit in PRODUITS_A_TESTER:
        sous = CATALOGUE[CATALOGUE["produit"] == produit]
        grille = fenetres(produit, ANNEES)
        if not len(sous):
            print(f"\n  {produit} — {SYM['ko']} aucune période publiée")
            continue

        par_periode = sous.groupby("periode")["tuile"].nunique().to_dict()
        attendues = [e for e, _, _ in grille]
        completes = [e for e in attendues if par_periode.get(e, 0) >= len(TUILES)]
        partielles = [e for e in attendues if 0 < par_periode.get(e, 0) < len(TUILES)]
        absentes = [e for e in attendues if par_periode.get(e, 0) == 0]

        print(f"\n  {produit} — {len(completes)}/{len(attendues)} période(s) complète(s) "
              f"({len(TUILES)} tuile(s) chacune)")
        if partielles:
            detail = ", ".join(f"{e} ({par_periode[e]}/{len(TUILES)})" for e in partielles[:8])
            print(f"     {SYM['att']} incomplètes : {detail}"
                  + (f" … (+{len(partielles)-8})" if len(partielles) > 8 else ""))
        if absentes:
            apercu = ", ".join(absentes[:12])
            print(f"     {SYM['ko']} absentes    : {apercu}"
                  + (f" … (+{len(absentes)-12})" if len(absentes) > 12 else ""))
        if not partielles and not absentes:
            print(f"     {SYM['ok']} série complète sur toute la grille demandée")

    print("\n  Extrait du catalogue (6 premières lignes) :")
    print(CATALOGUE.head(6)[["produit", "periode", "date", "tuile", "version",
                             "taille_mo", "fichier"]].to_string(index=False))
else:
    print(f"  {SYM['att']} Catalogue vide — rien à détailler.")

Deux lectures graphiques ferment l'étape. La première compare l'attendu au publié et chiffre le volume. La seconde est une **matrice de couverture** : une ligne par produit, une colonne par année, une case verte quand toutes les tuiles sont là. C'est la vue qui sert à décider — si une ligne est trouée, ce produit ne portera pas une série temporelle fiable pour votre pays, et il vaut mieux le savoir maintenant.

In [ ]:
# =============================================================================
#  ÉTAPE 7 ter — Lecture graphique de la disponibilité
# =============================================================================
if MPL_DISPO and len(INVENTAIRE) and INVENTAIRE["trouve"].sum() > 0:
    import numpy as np
    import matplotlib.pyplot as plt

    RAMPE = ["#00553A", "#00704A", "#00A86A", "#57BD92"]
    donnees = INVENTAIRE[INVENTAIRE["trouve"] > 0].reset_index(drop=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.6, 4.2), dpi=110)

    y = np.arange(len(donnees))
    ax1.barh(y, donnees["attendu"], color="#E8F5EF", edgecolor="#D5DED9",
             height=.62, label="attendu")
    ax1.barh(y, donnees["trouve"], color=[RAMPE[i % 4] for i in y],
             height=.62, label="publié")
    for i, r in donnees.iterrows():
        ax1.text(max(r["attendu"], r["trouve"]) * 1.03, i,
                 f"{int(r['trouve'])}/{int(r['attendu'])}  ({r['complet_pct']:.0f} %)",
                 va="center", fontsize=8.6, color="#231F20")
    ax1.set_yticks(y, donnees["produit"], fontsize=9.5, fontweight="bold")
    ax1.invert_yaxis()
    ax1.set_xlim(0, donnees[["attendu", "trouve"]].max().max() * 1.42)
    ax1.set_xlabel("Nombre de granules", fontsize=9, color="#5E6964")
    ax1.set_title("Granules publiés sur la grille demandée",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    ax1.legend(fontsize=8, frameon=False, loc="lower right")

    ax2.barh(y, donnees["volume_mo"] / 1024, color=[RAMPE[i % 4] for i in y], height=.62)
    for i, r in donnees.iterrows():
        ax2.text(r["volume_mo"] / 1024 * 1.03, i,
                 f"{r['volume_mo']/1024:.2f} Go  ·  {r['mo_par_fichier']:.0f} Mo/fichier",
                 va="center", fontsize=8.6, color="#231F20")
    ax2.set_yticks(y, donnees["produit"], fontsize=9.5, fontweight="bold")
    ax2.invert_yaxis()
    ax2.set_xlim(0, max(donnees["volume_mo"].max() / 1024, .01) * 1.75)
    ax2.set_xlabel("Volume à télécharger (Go)", fontsize=9, color="#5E6964")
    ax2.set_title("Poids du téléchargement complet",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)

    for ax in (ax1, ax2):
        ax.grid(axis="x", color="#E1E7E4", linewidth=.8)
        ax.set_axisbelow(True)
        ax.tick_params(labelsize=8.5, colors="#5E6964")
        for cote in ("top", "right", "left"):
            ax.spines[cote].set_visible(False)
        ax.spines["bottom"].set_color("#D5DED9")

    fig.suptitle(f"Black Marble — disponibilité pour {NOM_PAYS} ({len(TUILES)} tuile(s))",
                 fontsize=12.5, fontweight="bold", color="#00553A", y=1.04)
    fig.tight_layout()
    _fig = DOSSIERS["figures"] / f"disponibilite_{PAYS_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure enregistrée : {_fig}")
else:
    print(f"  {SYM['att']} Graphique ignoré (matplotlib absent ou catalogue vide).")

In [ ]:
# =============================================================================
#  ÉTAPE 7 quater — Matrice de couverture : produits × années
# =============================================================================
if MPL_DISPO and len(CATALOGUE):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap, BoundaryNorm

    produits = [p for p in PRODUITS_A_TESTER
                if len(CATALOGUE[CATALOGUE["produit"] == p])]
    matrice = np.zeros((len(produits), len(ANNEES)))
    textes = [["" for _ in ANNEES] for _ in produits]

    for i, p in enumerate(produits):
        sous = CATALOGUE[CATALOGUE["produit"] == p]
        par_periode = sous.groupby("periode")["tuile"].nunique().to_dict()
        for j, a in enumerate(ANNEES):
            f = fenetre_annee(p, a)
            if f is None:                       # date inexistante cette année-là
                matrice[i, j] = -1
                textes[i][j] = "—"
                continue
            n = par_periode.get(f[0], 0)
            matrice[i, j] = 2 if n >= len(TUILES) else (1 if n else 0)
            textes[i][j] = str(n) if n else ""

    couleurs = ListedColormap(["#F0F2F1", "#B83B2E", "#F5C242", "#00A86A"])
    norme = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], couleurs.N)

    fig, ax = plt.subplots(figsize=(max(7.5, 0.72 * len(ANNEES) + 3.2),
                                    1.05 * len(produits) + 2.1), dpi=110)
    ax.imshow(matrice, cmap=couleurs, norm=norme, aspect="auto")

    for i in range(len(produits)):
        for j in range(len(ANNEES)):
            if textes[i][j]:
                ax.text(j, i, textes[i][j], ha="center", va="center", fontsize=8.5,
                        color="#FFFFFF" if matrice[i, j] == 2 else "#231F20",
                        fontweight="bold")
    ax.set_xticks(np.arange(len(ANNEES)), [str(a) for a in ANNEES],
                  fontsize=8.5, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(produits)), produits, fontsize=9.5, fontweight="bold")
    ax.set_xticks(np.arange(-.5, len(ANNEES), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(produits), 1), minor=True)
    ax.grid(which="minor", color="#FFFFFF", linewidth=2.4)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(colors="#5E6964")
    for bord in ax.spines.values():
        bord.set_visible(False)

    reperes = [plt.Rectangle((0, 0), 1, 1, facecolor=c, edgecolor="#D5DED9")
               for c in ["#00A86A", "#F5C242", "#B83B2E", "#F0F2F1"]]
    ax.legend(reperes,
              [f"complet ({len(TUILES)} tuiles)", "partiel", "absent", "date inexistante"],
              loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=4,
              fontsize=8.5, frameon=False)

    jour = f"{int(JOUR_FIXE.split('-')[1])} {MOIS_FR[int(JOUR_FIXE.split('-')[0])]}"
    ax.set_title(f"Couverture par année — {NOM_PAYS} ({PAYS_ISO3})\n"
                 f"quotidiens : {jour} de chaque année  ·  mensuel : {MOIS_FR[MOIS_FIXE]}",
                 fontsize=11.5, fontweight="bold", color="#00553A", pad=14)
    fig.tight_layout()
    _fig = DOSSIERS["figures"] / f"couverture_{PAYS_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure enregistrée : {_fig}")
    print(f"  {SYM['pt']} Le chiffre dans chaque case est le nombre de tuiles trouvées.")
else:
    print(f"  {SYM['att']} Matrice ignorée (matplotlib absent ou catalogue vide).")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 08 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Téléchargement</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Plusieurs produits, reprise sur coupure, contrôle d'intégrité</div></div>

Un téléchargement scientifique n'est pas un `wget`. Quatre comportements sont attendus d'une chaîne d'acquisition sérieuse, et la fonction ci-dessous les implémente tous :

| Comportement | Pourquoi |
|---|---|
| **Reprise** sur fichier partiel (`Range: bytes=…`) | une coupure à 90 % ne doit pas coûter 90 % du travail |
| **Suivi manuel des redirections** | les redirections Earthdata perdent l'en-tête `Authorization` si on ne le réinjecte pas |
| **Vérification de la taille** annoncée | un fichier tronqué a l'air parfaitement normal sur le disque |
| **Ouverture de contrôle** en HDF5 | seule preuve qu'un `.h5` est réellement lisible |

### Choisir ce qui descend

`PRODUITS_A_TELECHARGER` est une **liste** : vous pouvez descendre le quotidien, le mensuel et l'annuel en une seule passe. Le mode décide du volume — `"aucun"` s'arrête à l'inventaire, `"echantillon"` prend les fichiers les plus récents de chaque produit dans la limite de `MAX_FICHIERS_PAR_PRODUIT`, `"complet"` prend toute la grille.

Le volume total est annoncé **avant** de commencer, produit par produit. Les fichiers sont rangés par pays, produit et année, et relancer la cellule est sans danger : ce qui est déjà valide est ignoré.

In [ ]:
# =============================================================================
#  ÉTAPE 8 — Téléchargement vérifié, résumable, idempotent
# =============================================================================
SESSION_DL = requests.Session()
SESSION_DL.headers.update({"User-Agent": f"{CLIENT_ID}/1.0"})
if JETON:
    SESSION_DL.headers.update({"Authorization": f"Bearer {JETON}"})


def fichier_valide(chemin, taille_min=1_000_000):
    """Le fichier existe, pèse un poids crédible, et s'ouvre réellement en HDF5."""
    if not chemin.exists() or chemin.stat().st_size < taille_min:
        return False
    if not H5PY_DISPO:
        return True                      # sans h5py on se contente du critère de taille
    try:
        import h5py
        with h5py.File(chemin, "r"):
            return True
    except Exception:
        return False


def verifier_jeton(url_test):
    """Demande les premiers octets d'un granule : diagnostic clair avant la boucle."""
    if not JETON:
        return False, "aucun jeton fourni"
    try:
        r = SESSION_DL.get(url_test, headers={"Range": "bytes=0-2047"},
                           stream=True, timeout=60)
        type_contenu = r.headers.get("content-type", "").lower()
        if "html" in type_contenu:
            return False, "réponse HTML — jeton invalide, expiré, ou EULA non acceptée"
        if r.status_code in (401, 403):
            return False, f"HTTP {r.status_code} — jeton refusé par Earthdata"
        if r.status_code not in (200, 206):
            return False, f"HTTP {r.status_code} inattendu"
        r.close()
        return True, "jeton accepté"
    except Exception as exc:
        return False, f"réseau indisponible ({exc})"


def telecharger(url, destination, essais=4):
    """Télécharge un granule. Renvoie ('ignore' | 'ok' | 'echec', octets)."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    partiel = destination.with_suffix(".h5.part")

    if fichier_valide(destination):
        return "ignore", destination.stat().st_size

    for essai in range(1, essais + 1):
        try:
            repris = partiel.stat().st_size if partiel.exists() else 0
            entetes = {"Range": f"bytes={repris}-"} if repris else {}

            courante, reponse = url, None
            for _ in range(10):                      # redirections, en gardant le jeton
                reponse = SESSION_DL.get(courante, headers=entetes, stream=True,
                                         timeout=300, allow_redirects=False)
                if reponse.status_code in (301, 302, 303, 307, 308):
                    courante = reponse.headers["Location"]
                    continue
                break

            if "html" in reponse.headers.get("content-type", "").lower():
                return "echec", 0                    # inutile d'insister : jeton en cause
            if reponse.status_code == 416:           # déjà complet côté serveur
                partiel.replace(destination)
                return "ok", destination.stat().st_size
            if reponse.status_code not in (200, 206):
                if essai < essais:
                    time.sleep(5 * essai)
                    continue
                return "echec", 0

            attendu = int(reponse.headers.get("content-length", 0))
            if reponse.status_code == 206:
                attendu += repris
            mode = "ab" if (repris and reponse.status_code == 206) else "wb"

            recus, jalon = repris, 0
            with open(partiel, mode) as sortie:
                for bloc in reponse.iter_content(1 << 20):   # 1 Mo
                    if not bloc:
                        continue
                    sortie.write(bloc)
                    recus += len(bloc)
                    if attendu:
                        avance = int(100 * recus / attendu)
                        if avance >= jalon + 25:
                            jalon = avance - avance % 25
                            print(f"        {jalon:3d} %", end="\r")
            if attendu:
                print(" " * 24, end="\r")            # efface la ligne de progression

            if attendu and partiel.stat().st_size < attendu:
                raise IOError(f"tronqué : {partiel.stat().st_size}/{attendu} octets")

            partiel.replace(destination)
            if not fichier_valide(destination):
                destination.unlink(missing_ok=True)
                raise IOError("fichier illisible en HDF5")
            return "ok", destination.stat().st_size

        except Exception as exc:
            print(f"        {SYM['att']} essai {essai}/{essais} : {exc}")
            if essai < essais:
                time.sleep(8 * essai)                # le .part est conservé pour la reprise

    return "echec", 0


# --- 8.1  Sélection : un sous-ensemble par produit ---------------------------
titre("Téléchargement")

selections = {}
if MODE_TELECHARGEMENT != "aucun" and len(CATALOGUE):
    for produit in PRODUITS_A_TELECHARGER:
        dispo = CATALOGUE[CATALOGUE["produit"] == produit].copy()
        if not len(dispo):
            print(f"  {SYM['att']} {produit} : aucun granule au catalogue, produit ignoré.")
            continue
        dispo = dispo.sort_values(["date", "tuile"], ascending=[False, True])
        retenu = dispo if MODE_TELECHARGEMENT == "complet" else dispo.head(MAX_FICHIERS_PAR_PRODUIT)
        selections[produit] = retenu
        ligne(f"{produit}",
              f"{len(retenu)} fichier(s) sur {len(dispo)} — "
              f"{retenu['taille_mo'].sum()/1024:.2f} Go — "
              f"périodes {retenu['periode'].min()} → {retenu['periode'].max()}", SYM["ok"])
else:
    print(f"  {SYM['pt']} Mode « {MODE_TELECHARGEMENT} » — aucun téléchargement demandé.")

a_telecharger = (pd.concat(selections.values(), ignore_index=True)
                 if selections else pd.DataFrame())
if len(a_telecharger):
    total_go = a_telecharger["taille_mo"].sum() / 1024
    print()
    ligne("TOTAL à télécharger", f"{len(a_telecharger)} fichier(s) — {total_go:.2f} Go",
          SYM["ok"] if total_go < 10 else SYM["att"])
    if total_go > 10:
        print(f"     {SYM['att']} Plus de 10 Go : vérifiez l'espace disque avant de poursuivre.")

# --- 8.2  Contrôle du jeton, puis boucle -------------------------------------
manifeste = []
if len(a_telecharger):
    ok_jeton, diagnostic = verifier_jeton(a_telecharger.iloc[0]["url"])
    ligne("Contrôle du jeton", diagnostic, SYM["ok"] if ok_jeton else SYM["ko"])

    if not ok_jeton:
        print(f"\n  {SYM['ko']} Téléchargement abandonné — l'inventaire reste exploitable.")
        print("     Vérifiez l'étape 3, puis relancez uniquement cette cellule.")
    else:
        for produit, retenu in selections.items():
            print(f"\n  {SYM['fl']} {produit} — {len(retenu)} fichier(s)")
            for rang, (_, g) in enumerate(retenu.iterrows(), 1):
                cible = (DOSSIERS["cache_h5"] / PAYS_ISO3 / g["produit"]
                         / str(g["annee"]) / g["fichier"])
                print(f"    [{rang}/{len(retenu)}] {g['periode']} · {g['tuile']} · "
                      f"{g['fichier']}  ({g['taille_mo']:.0f} Mo)")
                etat, octets = telecharger(g["url"], cible)
                symbole = {"ok": SYM["ok"], "ignore": SYM["pt"], "echec": SYM["ko"]}[etat]
                mot = {"ok": "téléchargé", "ignore": "déjà présent", "echec": "ÉCHEC"}[etat]
                print(f"        {symbole} {mot}" + (f" — {octets/1e6:.1f} Mo" if octets else ""))
                manifeste.append(dict(produit=g["produit"], periode=g["periode"],
                                      fichier=g["fichier"], tuile=g["tuile"],
                                      date=str(g["date"]), statut=etat,
                                      octets=octets, chemin=str(cible)))
                time.sleep(0.3)                      # courtoisie envers le serveur

if manifeste:
    MANIFESTE = pd.DataFrame(manifeste)
    _fm = DOSSIERS["catalogue"] / f"manifeste_{PAYS_ISO3}.csv"
    MANIFESTE.to_csv(_fm, index=False, encoding="utf-8-sig")
    reussis = int((MANIFESTE["statut"] != "echec").sum())
    print(f"\n  {SYM['ok']} {reussis}/{len(MANIFESTE)} fichier(s) disponibles sur le disque")
    for produit, sous in MANIFESTE.groupby("produit"):
        print(f"     {SYM['pt']} {produit} : {int((sous['statut'] != 'echec').sum())} fichier(s), "
              f"{sous['octets'].sum()/1e9:.2f} Go")
    print(f"  {SYM['ok']} Manifeste : {_fm}")
else:
    MANIFESTE = pd.DataFrame(columns=["produit", "periode", "fichier", "tuile",
                                      "date", "statut", "octets", "chemin"])

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">ÉTAPE 09 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Contrôle qualité</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Ouvrir un granule, lire la radiance, regarder l'image</div></div>

Un fichier téléchargé n'est pas encore une donnée exploitable. Trois vérifications concluent la collecte.

**On n'invente pas le chemin interne.** La structure HDF5 diffère d'un produit à l'autre (`VNP_Grid_DNB`, `VIIRS_Grid_DNB_2d`…). Plutôt que de coder un chemin en dur qui cassera au premier changement de produit, on parcourt l'arborescence et on cherche la variable par son nom.

**On applique les attributs.** Les valeurs sont stockées en entiers pour économiser de la place. La radiance physique s'obtient par `valeur × scale_factor + add_offset`, après avoir masqué les pixels marqués `_FillValue`. Sauter cette étape produit des cartes aux ordres de grandeur absurdes — c'est l'erreur la plus fréquente sur les données Black Marble.

**On regarde l'image.** La distribution de la radiance nocturne est extrêmement asymétrique : quelques pixels urbains écrasent tout le reste. Une échelle linéaire donne une image noire avec trois points blancs ; on applique donc une compression de dynamique avant d'afficher.

In [ ]:
# =============================================================================
#  ÉTAPE 9 — Contrôle qualité d'un granule téléchargé
# =============================================================================
fichiers_locaux = sorted((DOSSIERS["cache_h5"] / PAYS_ISO3).rglob("*.h5")) \
    if (DOSSIERS["cache_h5"] / PAYS_ISO3).exists() else []

titre("Contrôle qualité")

if not fichiers_locaux:
    print(f"  {SYM['att']} Aucun fichier .h5 sur le disque pour {PAYS_ISO3}.")
    print("     Passez MODE_TELECHARGEMENT à « echantillon » à l'étape 4 et relancez l'étape 8.")
elif not H5PY_DISPO:
    print(f"  {SYM['att']} h5py absent — installez-le puis relancez : pip install h5py")
    for f in fichiers_locaux[:5]:
        ligne(f.name, f"{f.stat().st_size/1e6:.1f} Mo", SYM["pt"])
else:
    import h5py
    import numpy as np

    ECHANTILLON = fichiers_locaux[-1]                 # le plus récent par tri alphabétique
    ligne("Fichiers sur le disque", str(len(fichiers_locaux)), SYM["ok"])
    ligne("Fichier examiné", ECHANTILLON.name, SYM["ok"])
    ligne("Taille", f"{ECHANTILLON.stat().st_size/1e6:.1f} Mo", SYM["pt"])

    # --- 9.1  Inventaire des variables, sans présumer du chemin ---------------
    variables = {}
    with h5py.File(ECHANTILLON, "r") as f:
        def visiter(nom, objet):
            if isinstance(objet, h5py.Dataset) and objet.ndim == 2:
                variables[nom] = objet.shape
        f.visititems(visiter)

        print(f"\n  Variables bidimensionnelles ({len(variables)}) :")
        for nom, forme in list(variables.items())[:14]:
            print(f"     {SYM['pt']} {nom.split('/')[-1]:<44} {forme}")
        if len(variables) > 14:
            print(f"     … et {len(variables)-14} autre(s)")

        # --- 9.2  Lecture de la variable principale du produit ---------------
        infos = _analyser_nom(ECHANTILLON.name) or {}
        souhaitee = CATALOGUE_PRODUITS.get(infos.get("produit", ""), {}).get("variable", "")
        chemin = next((n for n in variables if n.endswith(souhaitee)), None) \
            or next((n for n in variables
                     if any(c in n for c in ("NTL", "Radiance", "Composite"))), None)

        if chemin is None:
            print(f"\n  {SYM['att']} Variable de radiance introuvable dans ce fichier.")
            tableau = None
        else:
            def lire_attribut(valeur):
                """Un attribut HDF5 peut être un tableau, un scalaire ou des octets."""
                if isinstance(valeur, bytes):
                    return valeur.decode("utf-8", "ignore")
                plat = np.ravel(valeur)
                if plat.size == 1:
                    v = plat[0]
                    return v.decode("utf-8", "ignore") if isinstance(v, bytes) else v
                return valeur

            jeu = f[chemin]
            brut = jeu[:].astype("float64")
            attributs = {k: jeu.attrs[k] for k in jeu.attrs}
            facteur = float(lire_attribut(attributs.get("scale_factor", 1.0)))
            decalage = float(lire_attribut(attributs.get("add_offset", 0.0)))
            remplissage = attributs.get("_FillValue", None)

            tableau = brut.copy()
            if remplissage is not None:
                tableau[brut == float(lire_attribut(remplissage))] = np.nan
            tableau = tableau * facteur + decalage

            print(f"\n  Variable retenue : {chemin.split('/')[-1]}")
            ligne("  chemin interne", chemin, SYM["pt"])
            ligne("  dimensions", f"{jeu.shape[0]} × {jeu.shape[1]} pixels "
                                  f"(~{jeu.shape[0]*jeu.shape[1]/1e6:.1f} M)", SYM["pt"])
            ligne("  scale_factor / offset", f"{facteur:.6g} / {decalage:.6g}", SYM["pt"])
            ligne("  unité", str(lire_attribut(attributs.get("units", "nW·cm⁻²·sr⁻¹"))), SYM["pt"])

            valides = tableau[np.isfinite(tableau)]
            if valides.size:
                p50, p95, p99, p999 = np.percentile(valides, [50, 95, 99, 99.9])
                ligne("  pixels valides", f"{valides.size:,}".replace(",", " ")
                                          + f"  ({100*valides.size/tableau.size:.1f} %)", SYM["ok"])
                ligne("  médiane / p95", f"{p50:.3f} / {p95:.2f} nW·cm⁻²·sr⁻¹", SYM["pt"])
                ligne("  p99 / p99.9 / max", f"{p99:.1f} / {p999:.1f} / {valides.max():.1f}", SYM["pt"])
                ligne("  pixels « sombres » (<0.5)",
                      f"{100*(valides < 0.5).mean():.1f} % de la tuile", SYM["pt"])

In [ ]:
# =============================================================================
#  ÉTAPE 9 bis — Aperçu cartographique de la tuile
# =============================================================================
if fichiers_locaux and H5PY_DISPO and MPL_DISPO and 'tableau' in globals() and tableau is not None:
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, PowerNorm

    # Palette institutionnelle : nuit profonde → vert → or → blanc chaud
    PALETTE_NTL = LinearSegmentedColormap.from_list(
        "bad_ntl", ["#00100B", "#00553A", "#00A86A", "#F5C242", "#FFF6DC"])

    # Sous-échantillonnage : 2400×2400 pixels n'apportent rien à l'écran
    pas = max(1, tableau.shape[0] // 1200)
    vue = tableau[::pas, ::pas]
    plafond = float(np.nanpercentile(vue, 99.5)) or 1.0

    fig, (axa, axb) = plt.subplots(1, 2, figsize=(12.2, 5.2), dpi=110,
                                   gridspec_kw={"width_ratios": [1.25, 1]})

    image = axa.imshow(np.nan_to_num(vue), cmap=PALETTE_NTL,
                       norm=PowerNorm(gamma=0.32, vmin=0, vmax=plafond),
                       interpolation="nearest")
    axa.set_xticks([]); axa.set_yticks([])
    axa.set_title(f"{ECHANTILLON.name.split('.')[0]} · {infos.get('tuile','')} · "
                  f"{infos.get('date','')}",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    barre = fig.colorbar(image, ax=axa, fraction=.046, pad=.02)
    barre.set_label("Radiance (nW·cm⁻²·sr⁻¹) — échelle comprimée", fontsize=8.5, color="#5E6964")
    barre.ax.tick_params(labelsize=7.5, colors="#5E6964")

    valides = vue[np.isfinite(vue) & (vue > 0)]
    axb.hist(np.log10(valides + 0.01), bins=90, color="#00A86A", edgecolor="none")
    axb.set_xlabel("log₁₀(radiance + 0,01)", fontsize=9, color="#5E6964")
    axb.set_ylabel("Nombre de pixels", fontsize=9, color="#5E6964")
    axb.set_title("Distribution : l'asymétrie qui impose l'échelle logarithmique",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    axb.grid(axis="y", color="#E1E7E4", linewidth=.8); axb.set_axisbelow(True)
    axb.tick_params(labelsize=8.5, colors="#5E6964")
    for cote in ("top", "right"):
        axb.spines[cote].set_visible(False)
    for cote in ("bottom", "left"):
        axb.spines[cote].set_color("#D5DED9")

    fig.suptitle(f"Aperçu de contrôle — {NOM_PAYS} ({PAYS_ISO3})",
                 fontsize=12.5, fontweight="bold", color="#00553A", y=1.02)
    fig.tight_layout()
    _fig = DOSSIERS["figures"] / f"apercu_{PAYS_ISO3}_{infos.get('tuile','tuile')}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure enregistrée : {_fig}")
    print(f"  {SYM['pt']} Une tuile entière de 10°×10° : la découpe sur les frontières")
    print("     administratives se fait au notebook suivant (« Explorer et comprendre »).")
else:
    print(f"  {SYM['att']} Aperçu ignoré (pas de granule lu, ou matplotlib/h5py absent).")

<div style="border-left:6px solid #F5C242;background:#FFFDF6;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">BILAN</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Ce que la session a produit</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Un récapitulatif traçable, et un fichier de session pour le dépôt GitHub</div></div>

In [ ]:
# =============================================================================
#  BILAN — Inventaire des sorties et journal de session
# =============================================================================
duree_session = (dt.datetime.now() - SESSION_DEBUT).total_seconds()

titre("Bilan de session")
ligne("Pays", f"{NOM_PAYS} ({PAYS_ISO3})", SYM["ok"])
ligne("Tuiles", f"{len(TUILES)} — {', '.join(TUILES)}", SYM["pt"])
ligne("Produits testés", ", ".join(PRODUITS_A_TESTER), SYM["pt"])
ligne("Granules catalogués", f"{len(CATALOGUE)}", SYM["ok"] if len(CATALOGUE) else SYM["att"])
ligne("Granules téléchargés", f"{int((MANIFESTE['statut'] != 'echec').sum()) if len(MANIFESTE) else 0}",
      SYM["pt"])
ligne("Durée totale", f"{duree_session:.0f} s", SYM["pt"])

print("\n  Fichiers produits :")
total = 0
for dossier in ("catalogue", "figures"):
    for f in sorted(DOSSIERS[dossier].glob("*")):
        if f.is_file():
            total += f.stat().st_size
            print(f"     {SYM['pt']} {dossier}/{f.name:<44} {f.stat().st_size/1024:8.1f} Ko")
for f in sorted((DOSSIERS["cache_h5"] / PAYS_ISO3).rglob("*.h5")) if (DOSSIERS["cache_h5"] / PAYS_ISO3).exists() else []:
    total += f.stat().st_size
    print(f"     {SYM['pt']} h5_cache/…/{f.name:<38} {f.stat().st_size/1e6:8.1f} Mo")
print(f"\n     Total écrit : {total/1e6:.1f} Mo  dans  {RACINE}")

# Journal de session : reproductibilité et traçabilité pour le dépôt GitHub
session = dict(
    horodatage=SESSION_DEBUT.isoformat(timespec="seconds"),
    duree_s=round(duree_session, 1),
    plateforme=PLATEFORME, python=platform.python_version(),
    pays=dict(iso3=PAYS_ISO3, nom=NOM_PAYS, bbox=BBOX, tuiles=TUILES),
    produits=PRODUITS_A_TESTER,
    grille=dict(annees=ANNEES, jour_fixe=JOUR_FIXE, mois_fixe=MOIS_FIXE),
    mode_telechargement=MODE_TELECHARGEMENT,
    produits_telecharges=PRODUITS_A_TELECHARGER,
    inventaire=INVENTAIRE.drop(columns=["erreur"]).to_dict("records"),
    racine=str(RACINE),
)
_fj = DOSSIERS["journaux"] / f"session_{PAYS_ISO3}_{SESSION_DEBUT:%Y%m%d_%H%M%S}.json"
_fj.write_text(json.dumps(session, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"\n  {SYM['ok']} Journal de session : {_fj}")
print(f"\n  {SYM['fl']} Étape suivante : notebook « Explorer et comprendre » — masques de qualité,")
print("     statistiques zonales par région administrative, séries temporelles.")

---

## Dépannage

| Symptôme | Cause la plus probable | Correction |
|---|---|---|
| `Code ISO3 absent du répertoire` | code alpha-2 (`CI`) au lieu d'alpha-3 (`CIV`) | utilisez le code à trois lettres, ou renseignez `BBOX_MANUELLE` |
| Catalogue vide pour tous les produits | pas d'accès réseau sortant | sur Kaggle, activez *Internet* dans le panneau de droite ; en entreprise, exportez `HTTPS_PROXY` |
| `réponse HTML — jeton invalide` | jeton expiré, tronqué à la copie, ou EULA non acceptée | régénérez le jeton sur la page profil Earthdata et acceptez l'EULA *LAADS DAAC* |
| HTTP 401 / 403 au téléchargement | jeton absent de l'en-tête après redirection | déjà géré ici ; si l'erreur persiste, le jeton est refusé — régénérez-le |
| Le millésime annuel le plus récent manque | le composite n'est pas encore publié | comportement normal : les composites annuels paraissent avec plusieurs mois de décalage |
| Nuits manquantes en `VNP46A1` / `A2` | ces produits commencent le 19 janvier 2012 | ajustez la fenêtre, ou acceptez la lacune et documentez-la |
| Téléchargement interrompu | coupure réseau | relancez la cellule : le `.part` est repris là où il s'était arrêté |
| `fichier illisible en HDF5` | téléchargement tronqué et non détecté | le fichier est supprimé automatiquement ; relancez |
| Plus de place sur Colab | disque de session saturé | montez Google Drive et redéfinissez `NTL_HOME` |
| Carte entièrement noire | `scale_factor` non appliqué, ou échelle linéaire | vérifiez l'étape 9.2 : l'échelle doit être comprimée |
| `UnicodeEncodeError` en console Windows | console héritée en cp1252 | déjà géré ; sinon `chcp 65001` avant de lancer Python |


<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ Ce que ce notebook ne fait pas — et où cela se fait</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">La collecte s'arrête à la tuile brute. La découpe sur les frontières nationales et régionales, l'application des masques de qualité, les statistiques zonales et la validation face aux statistiques officielles relèvent des parties 2 à 4 du Jour 4. Une tuile Black Marble est un carré de 10° de côté&nbsp;: elle déborde largement de votre pays, et c'est normal.</div></div>

---

## Références et licence des données

**Produits et documentation**

- Román, M. O. *et al.* (2018). *NASA's Black Marble nighttime lights product suite*. **Remote Sensing of Environment**, 210, 113–143. <https://doi.org/10.1016/j.rse.2018.03.017>
- *Black Marble User Guide*, NASA LAADS DAAC — <https://ladsweb.modaps.eosdis.nasa.gov>
- DOI des produits : `10.5067/VIIRS/VNP46A1.001` · `…VNP46A2.001` · `…VNP46A3.001` · `…VNP46A4.001`

**Services utilisés**

- NASA CMR — recherche de granules : <https://cmr.earthdata.nasa.gov/search/granules.json>
- Earthdata Login — comptes et jetons : <https://urs.earthdata.nasa.gov>

**Emprises nationales**

- Rectangles englobants dérivés de Natural Earth (domaine public). À des fins de requête uniquement : ce ne sont pas des frontières officielles et ils n'engagent personne sur le tracé des limites.

**Licence des données**

Les produits Black Marble relèvent de la politique de données ouvertes de la NASA : réutilisation libre, y compris commerciale, sous réserve de citation. Pour une publication officielle, citez le produit, sa version, son DOI et la date d'extraction — que le journal de session enregistre pour vous.

**Citation suggérée de ce notebook**

> Banque africaine de développement (2026). *Lumières nocturnes : collecte des données NASA Black Marble*. Groupe technique spécialisé n° 17 — Questions émergentes, atelier « Emerging Issues, Emerging Practice ». Notebook, Jour 4, partie 1.


<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 60%,#00A86A 100%);border-radius:18px;padding:26px 34px;margin-top:22px;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;text-align:center;"><div style="color:#F5C242;font-size:11.5px;letter-spacing:3px;font-weight:700;">BOÎTE À OUTILS DATA — BANQUE AFRICAINE DE DÉVELOPPEMENT</div><div style="color:#fff;font-size:1.18em;font-weight:700;margin-top:8px;">Un code ISO3 en entrée &nbsp;·&nbsp; quatre produits testés &nbsp;·&nbsp; un inventaire reproductible en sortie</div><div style="color:#CFEEDE;font-size:.94em;margin-top:10px;">Jour 4 &nbsp;·&nbsp; Partie 1 «&nbsp;Collecter&nbsp;» &nbsp;→&nbsp; Partie 2 «&nbsp;Explorer et comprendre&nbsp;»</div><div style="margin:16px auto 0 auto;height:3px;width:110px;background:#F5C242;border-radius:2px;"></div></div>